# Anem a buscar una mica de non-eq phys dins de tot això

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Llegim el csv i afegim la temperatura prime

In [ ]:
prepared_path = Path("../../data/df_dynamics_with_oof.csv")
df_prepared = pd.read_csv(prepared_path)



K_MIX, K_SUN, K_WIND = 3.0, 9.5, 16.0

df1 = df_prepared.copy()

df1 = df1.dropna(
    subset=[
        "ID", "stop_idx", "walk_id", "comfort3",
        "<T-T_fixed+<T>>", "sun", "wind"
    ]
).copy()

df1["T_corr"] = df1["<T-T_fixed+<T>>"]

df1["f_mix"] = (df1["sun"] == "In a mixture of sun and shadow").astype(float)
df1["f_sun"] = (df1["sun"] == "In full sun").astype(float)

WIND_MAP = {
    "It's not windy": 0.0,
    "A very light wind": 0.25,
    "A light wind": 0.5,
    "A moderate wind": 0.75,
    "A strong wind": 1.0,
}

df1["wind_sc"] = df1["wind"].map(WIND_MAP)

df1["T_env"] = (
    df1["T_corr"]
    + K_MIX * df1["f_mix"]
    + K_SUN * df1["f_sun"]
    - K_WIND * df1["wind_sc"]
)
print(df1.shape)
df1.columns

df_prepared= df1.copy()

## Definim els estats amb el Confort3_Option1

In [ ]:
state_order = ["comfortable", "neutral", "uncomfortable"]
# state_order = [
#     "Very comfortable", "Comfortable", "Slightly comfortable",
#     "Neutral",
#     "Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"
# ]

state_num_map = {s: i for i, s in enumerate(state_order)}

df = df1.copy()

# Important: ordenar abans de fer shifts
df = df.sort_values(["ID", "walk_id", "stop_idx"]).copy()

df["state"] = df["comfort3"]
df["state_num"] = df["state"].map(state_num_map)

# Temperatures al stop actual
df["T_corr_start"] = df["T_corr"]
df["T_env_start"] = df["T_env"]

# Temperatures al stop de destinació
df["T_corr_end"] = df.groupby(["ID", "walk_id"])["T_corr"].shift(-1)
df["T_env_end"] = df.groupby(["ID", "walk_id"])["T_env"].shift(-1)

# Temperatures mitjanes de la transició
df["T_corr_mean"] = 0.5 * (df["T_corr_start"] + df["T_corr_end"])
df["T_env_mean"] = 0.5 * (df["T_env_start"] + df["T_env_end"])


# Estat següent
df["next_state"] = df.groupby(["ID", "walk_id"])["state"].shift(-1)
df["next_state_num"] = df.groupby(["ID", "walk_id"])["state_num"].shift(-1)
df["next_stop_idx"] = df.groupby(["ID", "walk_id"])["stop_idx"].shift(-1)

# Ens assegurem que només mirem transicions consecutives
df["step_gap"] = df["next_stop_idx"] - df["stop_idx"]

transitions = df[df["step_gap"] == 1].copy()

# Si vols limitar a les primeres transicions
transitions = transitions[transitions["stop_idx"] <= 5].copy()

# Canvi d'estat
transitions["delta"] = transitions["next_state_num"] - transitions["state_num"]

transitions

# Occupancy per stop.
M'agradaria afegir-li el núm de vots associats a cada parada

In [ ]:
occ = (
    df.groupby("stop_idx")["state"]
      .value_counts(normalize=True)
      .rename("p")
      .reset_index()
)

occ.to_csv("csvs_proces/occ_by_stop.csv", index=False)
# Afegir al df el numero de vots per stop
occ



In [ ]:
occ_pivot = (
    occ.pivot(index="stop_idx", columns="state", values="p")
    .reindex(columns=state_order)
    .fillna(0)
)

plt.figure(figsize=(8, 5))
bottom = np.zeros(len(occ_pivot))

for col in occ_pivot.columns:
    plt.bar(
        occ_pivot.index,
        occ_pivot[col].values,
        bottom=bottom,
        label=col
    )
    bottom += occ_pivot[col].values

plt.xlabel("Stop")
plt.ylabel("Proporció")
plt.title("Ocupació dels estats per stop")
plt.legend()
plt.tight_layout()
plt.show()

# Matriu de transició

In [ ]:
cm_counts = pd.crosstab(
    transitions["state"],
    transitions["next_state"]
).reindex(index=state_order, columns=state_order, fill_value=0)

cm_prob = cm_counts.div(cm_counts.sum(axis=1), axis=0)

cm_prob.to_csv("csvs_proces/cm_prob.csv", index=False)
cm_prob

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_prob.values, aspect="auto", cmap="Reds", vmin=0, vmax=1)

ax.set_xticks(range(len(state_order)))
ax.set_yticks(range(len(state_order)))
ax.set_xticklabels(state_order, rotation=30, ha="right")
ax.set_yticklabels(state_order)

ax.set_xlabel("Estat següent")
ax.set_ylabel("Estat actual")
ax.set_title("Matriu de transició (probabilitats per fila)")

for i in range(cm_prob.shape[0]):
    for j in range(cm_prob.shape[1]):
        val = cm_prob.iloc[i, j]
        ax.text(
            j, i,
            f"{val:.3f}",
            ha="center", va="center",
            color="black" if val < 0.6 else "white"
        )

fig.colorbar(im, ax=ax, label="Probabilitat")
plt.tight_layout()
plt.show()

# Proba per a fer un escombrat

In [ ]:
def build_transitions_for_trap_sweep(
    df_prepared,
    state_col = "comfort3_option1",
    t_col = "T_env",
    hdx_col = "<HDX-HDX_fixed+<HDX>>",
    max_stop=5,
):
    df = df_prepared.copy()

    sort_cols = [c for c in ["ID", "walk_id", "stop_idx", "row_id"] if c in df.columns]
    group_cols = [c for c in ["ID", "walk_id"] if c in df.columns]

    df = df.sort_values(sort_cols).reset_index(drop=True)
    g = df.groupby(group_cols, sort=False)

    df["state"] = df[state_col]
    df["state_num"] = df["state"].map(state_num_map)

    df["next_state"] = g["state"].shift(-1)
    df["next_state_num"] = g["state_num"].shift(-1)
    df["next_stop_idx"] = g["stop_idx"].shift(-1)

    df["step_gap"] = df["next_stop_idx"] - df["stop_idx"]

    # T variables
    if t_col in df.columns:
        df["T_t"] = df[t_col]
        df["T_t1"] = g[t_col].shift(-1)
        df["T_mean"] = 0.5 * (df["T_t"] + df["T_t1"])
        df["T_delta"] = df["T_t1"] - df["T_t"]

    # HDX variables
    if hdx_col in df.columns:
        df["HDX_t"] = df[hdx_col]
        df["HDX_t1"] = g[hdx_col].shift(-1)
        df["HDX_mean"] = 0.5 * (df["HDX_t"] + df["HDX_t1"])
        df["HDX_delta"] = df["HDX_t1"] - df["HDX_t"]

    transitions = df[
        (df["step_gap"] == 1) &
        (df["stop_idx"] <= max_stop) &
        (df["next_stop_idx"] <= max_stop)
    ].copy()

    transitions = transitions.dropna(subset=["state", "next_state"]).copy()
    transitions["delta_state"] = transitions["next_state_num"] - transitions["state_num"]

    return transitions

In [ ]:
transitions = build_transitions_for_trap_sweep(
    df_prepared,
    max_stop=5
)

transitions.shape

In [ ]:
def make_event_df(transitions, metric):
    df = transitions.copy()

    if metric == "vu_trap":
        # P(VU -> VU)
        sub = df[df["state"] == "very uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "very uncomfortable").astype(int)

    elif metric == "u_persistence":
        # P(U -> U)
        sub = df[df["state"] == "uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "uncomfortable").astype(int)

    elif metric == "u_adverse_persistence":
        # P(U -> U or VU)
        sub = df[df["state"] == "uncomfortable"].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ]).astype(int)

    elif metric == "discomfort_basin_persistence":
        # P(next in {U,VU} | current in {U,VU})
        sub = df[df["state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ])].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ]).astype(int)

    elif metric == "recovery_from_discomfort_basin":
        # P(next = CN | current in {U,VU})
        sub = df[df["state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ])].copy()
        sub["event"] = (sub["next_state"] == "comfortable-neutral").astype(int)

    elif metric == "onset_of_discomfort":
        # P(next in {U,VU} | current = CN)
        sub = df[df["state"] == "comfortable-neutral"].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
            "very uncomfortable"
        ]).astype(int)

    else:
        raise ValueError(f"Unknown metric: {metric}")

    return sub

for metric in [
    "vu_trap",
    "u_persistence",
    "u_adverse_persistence",
    "discomfort_basin_persistence",
    "recovery_from_discomfort_basin",
    "onset_of_discomfort"
]:
    sub = make_event_df(transitions, metric)



KEEP = {
    'row_id', 'ID', 'walk_id', 'space_code',
    'stop_idx', 'next_stop_idx', 'state', 'state_num', 'next_state', 'next_state_num',
    'thermal_comfort','comfort3_option1', 'thermal_comfort_walking', 'thermal_sensation',
    'gender', 'age', 'spent_time',
    '<T-T_fixed+<T>>', '<HDX-HDX_fixed+<HDX>>', 
    'sun', 'wind', 'SVF_1.5m', 'NDVI_1.5m', 'climate_shelter', 'IVAC',
    'T_t', 'T_t1', 'T_mean', 'T_delta', 
    'HDX_t', 'HDX_t1', 'HDX_mean', 'HDX_delta', 'hdx_bin',
    'city', 'place', 'date'
}

sub = sub[[col for col in sub.columns if col in KEEP]]


sub.columns

sub.to_csv(f"saved_df/transitions_events.csv", index=False)

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2))) / denom

    return center - half, center + half

## i ara ja definim el sweep

In [ ]:
def sweep_quantile_bins(
    transitions,
    x_col,
    metric,
    n_bins=6,
    min_at_risk=20,
):
    event_df = make_event_df(transitions, metric)
    event_df = event_df.dropna(subset=[x_col, "event"]).copy()

    event_df["x_bin"] = pd.qcut(
        event_df[x_col],
        q=n_bins,
        duplicates="drop"
    )

    rows = []

    for xbin, sub in event_df.groupby("x_bin", observed=True):
        n = len(sub)
        k = int(sub["event"].sum())

        if n < min_at_risk:
            continue

        p = k / n
        lo, hi = wilson_ci(k, n)

        rows.append({
            "metric": metric,
            "x_col": x_col,
            "x_bin": xbin,
            "x_min": sub[x_col].min(),
            "x_max": sub[x_col].max(),
            "x_mean": sub[x_col].mean(),
            "x_median": sub[x_col].median(),
            "n_at_risk": n,
            "events": k,
            "prob": p,
            "ci_low": lo,
            "ci_high": hi,
            "score_ci_low": lo,
        })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values(
        ["score_ci_low", "prob", "n_at_risk"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

In [ ]:
metrics_to_test = [
    #"vu_trap",
    #"u_persistence",
    #"u_adverse_persistence",
    #"discomfort_basin_persistence",
    "recovery_from_discomfort_basin",
    "onset_of_discomfort",
]

x_cols_to_test = [
    #"T_mean",
    "T_t1",
    #"HDX_mean",
    "HDX_t1",
    #"T_delta",
    #"HDX_delta",
]

bin_results = []

for metric in metrics_to_test:
    for x_col in x_cols_to_test:
        if x_col in transitions.columns:
            tmp = sweep_quantile_bins(
                transitions,
                x_col=x_col,
                metric=metric,
                n_bins=10,
                min_at_risk=20
            )
            bin_results.append(tmp)

bin_results = pd.concat(bin_results, ignore_index=True)

bin_results.to_csv("markov_results/trap_sweep_bins_T_HDX.csv", index=False)

bin_results.head(30)

# Probem a mirar coses per edat

In [ ]:
import re
import numpy as np
import pandas as pd


def parse_age_to_numeric(age_value):
    """
    Convert age entries into a representative numeric age.

    Examples
    --------
    12          -> 12
    "12"        -> 12
    "<=12"      -> 12
    "13-15"     -> 14
    "16-34"     -> 25
    "65+"       -> 65
    ">65"       -> 65
    NaN         -> NaN
    """

    if pd.isna(age_value):
        return np.nan

    s = str(age_value).strip()

    if s == "":
        return np.nan

    # Normalize separators
    s = s.replace("–", "-").replace("—", "-")

    # Extract all numbers
    nums = re.findall(r"\d+\.?\d*", s)

    if len(nums) == 0:
        return np.nan

    nums = [float(x) for x in nums]

    # If range, use midpoint
    if len(nums) >= 2:
        return np.mean(nums[:2])

    # If single number, use it directly
    return nums[0]


def add_binary_age_group(df, age_col="age", age_threshold=16):
    """
    Add:
    - age_numeric
    - age_binary

    age_binary:
    - <= threshold
    - > threshold
    """

    df = df.copy()

    df["age_numeric"] = df[age_col].apply(parse_age_to_numeric)

    df["age_binary"] = np.where(
        df["age_numeric"] > age_threshold,
        f">{age_threshold}",
        f"<={age_threshold}"
    )

    df.loc[df["age_numeric"].isna(), "age_binary"] = np.nan

    return df

In [ ]:
AGE_THRESHOLD = 14

transitions = add_binary_age_group(
    transitions,
    age_col="age",
    age_threshold=AGE_THRESHOLD
)

transitions[["age", "age_numeric", "age_binary"]].drop_duplicates().sort_values("age_numeric")

In [ ]:
transitions.groupby("age_binary", dropna=False).agg(
    n_transitions=("state", "count"),
    n_ID=("ID", "nunique"),
    n_walks=("walk_id", "nunique")
).reset_index()

In [ ]:
from pathlib import Path

OUT_DIR = Path("markov_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

bootstrap_threshold_results_age_above = []

for age_group, transitions_age in transitions.groupby("age_binary", dropna=False):

    if pd.isna(age_group):
        continue

    print("Running age group:", age_group, "n =", len(transitions_age))

    for metric in metrics_to_test:
        for x_col in x_cols_to_test:

            if x_col in transitions_age.columns:

                print(f"Running ABOVE | age_binary={age_group} | metric={metric} | x_col={x_col}")

                tmp = bootstrap_threshold_sweep(
                    transitions=transitions_age,
                    x_col=x_col,
                    metric=metric,
                    direction="above",
                    n_grid=60,
                    min_at_risk=20,
                    q_low=0.05,
                    q_high=0.95,
                    n_boot=1000,
                    seed=123,
                    cluster_cols=("ID", "walk_id"),
                )

                if not tmp.empty:
                    tmp["age_binary"] = age_group
                    tmp["age_threshold"] = AGE_THRESHOLD
                    bootstrap_threshold_results_age_above.append(tmp)

if len(bootstrap_threshold_results_age_above) == 0:
    raise ValueError(
        "No bootstrap threshold results were generated for age_binary ABOVE. "
        "Check min_at_risk, metrics, x_cols, or age groups."
    )

bootstrap_threshold_results_age_above = pd.concat(
    bootstrap_threshold_results_age_above,
    ignore_index=True
)

bootstrap_threshold_results_age_above.to_csv(
    OUT_DIR / f"age_binary_{AGE_THRESHOLD}_bootstrap_threshold_sweep_T_HDX_above.csv",
    index=False
)

bootstrap_threshold_results_age_above.head(20)

In [ ]:
bootstrap_threshold_results_age_below = []

for age_group, transitions_age in transitions.groupby("age_binary", dropna=False):

    if pd.isna(age_group):
        continue

    print("Running age group:", age_group, "n =", len(transitions_age))

    for metric in metrics_to_test:
        for x_col in x_cols_to_test:

            if x_col in transitions_age.columns:

                print(f"Running BELOW | age_binary={age_group} | metric={metric} | x_col={x_col}")

                tmp = bootstrap_threshold_sweep(
                    transitions=transitions_age,
                    x_col=x_col,
                    metric=metric,
                    direction="below",
                    n_grid=60,
                    min_at_risk=20,
                    q_low=0.05,
                    q_high=0.95,
                    n_boot=1000,
                    seed=123,
                    cluster_cols=("ID", "walk_id"),
                )

                if not tmp.empty:
                    tmp["age_binary"] = age_group
                    tmp["age_threshold"] = AGE_THRESHOLD
                    bootstrap_threshold_results_age_below.append(tmp)

if len(bootstrap_threshold_results_age_below) == 0:
    raise ValueError(
        "No bootstrap threshold results were generated for age_binary BELOW. "
        "Check min_at_risk, metrics, x_cols, or age groups."
    )

bootstrap_threshold_results_age_below = pd.concat(
    bootstrap_threshold_results_age_below,
    ignore_index=True
)

bootstrap_threshold_results_age_below.to_csv(
    OUT_DIR / f"age_binary_{AGE_THRESHOLD}_bootstrap_threshold_sweep_T_HDX_below.csv",
    index=False
)

bootstrap_threshold_results_age_below.head(20)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

CSV_PATH = Path(f"markov_results/age_binary_{AGE_THRESHOLD}_bootstrap_threshold_sweep_T_HDX_above.csv")

OUT_DIR = Path("markov_results/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

x_col = "T_t1"
direction = "above"

metric_onset = "onset_of_discomfort"
metric_recovery = "recovery_from_discomfort_basin"

age_order = [
    f"<={AGE_THRESHOLD}",
    f">{AGE_THRESHOLD}",
]

T_ref = 29.0

# ============================================================
# LOAD DATA
# ============================================================

df_age = pd.read_csv(CSV_PATH)

df_age = df_age[
    (df_age["x_col"] == x_col) &
    (df_age["direction"] == direction) &
    (df_age["metric"].isin([metric_onset, metric_recovery]))
].copy()

df_age["age_binary"] = df_age["age_binary"].astype(str)

df_age = df_age[df_age["age_binary"].isin(age_order)].copy()

print(df_age.groupby(["age_binary", "metric"]).size())

In [ ]:
def build_rd_for_group(
    df_group,
    group_col,
    metric_onset="onset_of_discomfort",
    metric_recovery="recovery_from_discomfort_basin",
):
    onset = (
        df_group[df_group["metric"] == metric_onset]
        .sort_values("threshold")
        .copy()
    )

    recovery = (
        df_group[df_group["metric"] == metric_recovery]
        .sort_values("threshold")
        .copy()
    )

    if onset.empty or recovery.empty:
        return pd.DataFrame()

    t_min = max(onset["threshold"].min(), recovery["threshold"].min())
    t_max = min(onset["threshold"].max(), recovery["threshold"].max())

    grid = onset.loc[
        onset["threshold"].between(t_min, t_max),
        "threshold"
    ].to_numpy()

    if len(grid) == 0:
        return pd.DataFrame()

    out = pd.DataFrame({"threshold": grid})

    cols = [
        "prob_observed",
        "median",
        "q025",
        "q975",
        "n_at_risk",
        "events",
    ]

    for col in cols:
        if col not in onset.columns or col not in recovery.columns:
            raise ValueError(f"Missing column in bootstrap CSV: {col}")

        out[f"onset_{col}"] = np.interp(
            grid,
            onset["threshold"].to_numpy(),
            onset[col].to_numpy()
        )

        out[f"recovery_{col}"] = np.interp(
            grid,
            recovery["threshold"].to_numpy(),
            recovery[col].to_numpy()
        )

    out["R_D_observed"] = (
        out["onset_prob_observed"] /
        out["recovery_prob_observed"]
    )

    out["R_D_median"] = (
        out["onset_median"] /
        out["recovery_median"]
    )

    out["R_D_q025"] = (
        out["onset_q025"] /
        out["recovery_q975"]
    )

    out["R_D_q975"] = (
        out["onset_q975"] /
        out["recovery_q025"]
    )

    out["n_min"] = np.minimum(
        out["onset_n_at_risk"],
        out["recovery_n_at_risk"]
    )

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(
        subset=["R_D_median", "R_D_q025", "R_D_q975"]
    )

    return out

In [ ]:
rd_age_list = []

for age_group in age_order:
    sub_g = df_age[df_age["age_binary"] == age_group].copy()

    rd_g = build_rd_for_group(
        sub_g,
        group_col="age_binary",
        metric_onset=metric_onset,
        metric_recovery=metric_recovery,
    )

    if not rd_g.empty:
        rd_g["age_binary"] = age_group
        rd_g["age_threshold"] = AGE_THRESHOLD
        rd_age_list.append(rd_g)

if len(rd_age_list) == 0:
    raise ValueError(
        "No R_D data could be built. Check metrics, x_col, direction, or age_binary values."
    )

rd_age = pd.concat(rd_age_list, ignore_index=True)

rd_age.to_csv(
    OUT_DIR / f"RD_T_next_by_age_binary_{AGE_THRESHOLD}_from_bootstrap.csv",
    index=False
)

display(rd_age.head())

In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, ax = plt.subplots(figsize=(7.4, 4.7))

for age_group in age_order:
    sub = rd_age[rd_age["age_binary"] == age_group].sort_values("threshold")

    if sub.empty:
        continue

    x = sub["threshold"].to_numpy(dtype=float)
    y = sub["R_D_median"].to_numpy(dtype=float)
    lo = sub["R_D_q025"].to_numpy(dtype=float)
    hi = sub["R_D_q975"].to_numpy(dtype=float)

    ax.fill_between(
        x,
        lo,
        hi,
        alpha=0.18,
        linewidth=0
    )

    ax.plot(
        x,
        y,
        marker="o",
        markersize=3.5,
        linewidth=1.9,
        label=age_group
    )

ax.axhline(
    1,
    linestyle="--",
    linewidth=1.2,
    color="black",
    alpha=0.75
)

ax.axvline(
    T_ref,
    linestyle=":",
    linewidth=1.2,
    color="black",
    alpha=0.75
)

ax.set_ylim(0.1, 3)
ax.text(
    T_ref + 0.06,
    ax.get_ylim()[1] * 0.93,
    f"{T_ref:.0f}°C",
    fontsize=10,
    va="top"
)


ax.set_xlabel(r"Next-stop temperature threshold, $T_{t+1}^{*}$ [°C]")

ax.set_ylabel(
    r"$R_D(T^*) = "
    r"\frac{P(\mathrm{onset\ of\ discomfort}\mid T_{t+1}\geq T^*)}"
    r"{P(\mathrm{recovery\ from\ discomfort}\mid T_{t+1}\geq T^*)}$"
)

ax.set_title(
    rf"Discomfort pressure ratio by age group, conditioned on $T_{{t+1}}$"
)

ax.grid(axis="y", alpha=0.25)
ax.legend(title="Age group", frameon=True)

fig.tight_layout()

fig.savefig(
    OUT_DIR / f"RD_T_next_by_age_binary_{AGE_THRESHOLD}_publication.png",
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    OUT_DIR / f"RD_T_next_by_age_binary_{AGE_THRESHOLD}_publication.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
bin_results[
    (bin_results["metric"] == "discomfort_basin_persistence") &
    (bin_results["x_col"] == "HDX_mean")
].sort_values(["score_ci_low", "prob"], ascending=False)

In [ ]:
def plot_bin_sweep(bin_results, metric, x_col, title=None):
    sub = bin_results[
        (bin_results["metric"] == metric) &
        (bin_results["x_col"] == x_col)
    ].copy()

    if sub.empty:
        print(f"No results for {metric} / {x_col}")
        return

    sub = sub.sort_values("x_mean")

    x = sub["x_mean"].to_numpy()
    y = sub["prob"].to_numpy()

    yerr_low = y - sub["ci_low"].to_numpy()
    yerr_high = sub["ci_high"].to_numpy() - y

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.errorbar(
        x,
        y,
        yerr=[yerr_low, yerr_high],
        fmt="o-",
        capsize=4
    )

    for _, row in sub.iterrows():
        ax.text(
            row["x_mean"],
            row["prob"] + 0.015,
            f"n={int(row['n_at_risk'])}",
            ha="center",
            fontsize=8
        )

    ax.set_xlabel(x_col)
    ax.set_ylabel(f"P({metric})")

    if title is None:
        title = f"{metric} as a function of {x_col}"

    ax.set_title(title)
    ax.grid(alpha=0.3)

    ymin = max(0, sub["ci_low"].min() - 0.05)
    ymax = min(1, sub["ci_high"].max() + 0.05)
    ax.set_ylim(ymin, ymax)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_bin_sweep(
    bin_results,
    metric="vu_trap",
    x_col="HDX_mean",
    title="Very-uncomfortable trap probability vs HDX"
)

plot_bin_sweep(
    bin_results,
    metric="discomfort_basin_persistence",
    x_col="HDX_mean",
    title="Discomfort-basin persistence vs HDX"
)

plot_bin_sweep(
    bin_results,
    metric="recovery_from_discomfort_basin",
    x_col="HDX_mean",
    title="Recovery from discomfort basin vs HDX"
)

plot_bin_sweep(
    bin_results,
    metric="discomfort_basin_persistence",
    x_col="T_mean",
    title="Discomfort-basin persistence vs T"
)

plot_bin_sweep(
    bin_results,
    metric="recovery_from_discomfort_basin",
    x_col="T_mean",
    title="Recovery from discomfort basin vs T"
)

## si fem l'esombrat per treshold

In [ ]:
def sweep_thresholds(
    transitions,
    x_col,
    metric,
    direction="above",
    n_grid=50,
    min_at_risk=20,
    q_low=0.05,
    q_high=0.95,
):
    event_df = make_event_df(transitions, metric)
    event_df = event_df.dropna(subset=[x_col, "event"]).copy()

    x = event_df[x_col].to_numpy()
    thresholds = np.quantile(
        x,
        np.linspace(q_low, q_high, n_grid)
    )

    rows = []

    for thr in thresholds:
        if direction == "above":
            sub = event_df[event_df[x_col] >= thr].copy()
            condition = f"{x_col} >= {thr:.3f}"
        elif direction == "below":
            sub = event_df[event_df[x_col] <= thr].copy()
            condition = f"{x_col} <= {thr:.3f}"
        else:
            raise ValueError("direction must be 'above' or 'below'")

        n = len(sub)
        if n < min_at_risk:
            continue

        k = int(sub["event"].sum())
        p = k / n
        lo, hi = wilson_ci(k, n)

        rows.append({
            "metric": metric,
            "x_col": x_col,
            "direction": direction,
            "threshold": thr,
            "condition": condition,
            "n_at_risk": n,
            "events": k,
            "prob": p,
            "ci_low": lo,
            "ci_high": hi,
            "score_ci_low": lo,
        })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values(
        ["score_ci_low", "prob", "n_at_risk"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

In [ ]:
threshold_results = []

for metric in metrics_to_test:
    for x_col in ["T_mean", "T_t1"]:
        if x_col in transitions.columns:
            tmp = sweep_thresholds(
                transitions,
                x_col=x_col,
                metric=metric,
                direction="above",
                n_grid=60,
                min_at_risk=20,
                q_low=0.05,
                q_high=0.95,
            )
            threshold_results.append(tmp)

threshold_results = pd.concat(threshold_results, ignore_index=True)

threshold_results.to_csv("markov_newT/trap_sweep_thresholds_T.csv", index=False)

threshold_results.head(30)

In [ ]:
threshold_results[
    (threshold_results["metric"] == "vu_trap") &
    (threshold_results["x_col"] == "HDX_mean")
].head(10)

In [ ]:
def plot_threshold_sweep(threshold_results, metric, x_col, direction="above", title=None):
    sub = threshold_results[
        (threshold_results["metric"] == metric) &
        (threshold_results["x_col"] == x_col) &
        (threshold_results["direction"] == direction)
    ].copy()

    if sub.empty:
        print(f"No results for {metric} / {x_col}")
        return

    sub = sub.sort_values("threshold")

    x = sub["threshold"].to_numpy()
    y = sub["prob"].to_numpy()

    yerr_low = y - sub["ci_low"].to_numpy()
    yerr_high = sub["ci_high"].to_numpy() - y

    fig, ax1 = plt.subplots(figsize=(8, 5))

    ax1.errorbar(
        x,
        y,
        yerr=[yerr_low, yerr_high],
        fmt="o-",
        capsize=4,
        label=f"P({metric} | {x_col} {direction})"
    )

    ax1.set_xlabel(f"{x_col} threshold")
    ax1.set_ylabel(f"P({metric})")
    ax1.grid(alpha=0.3)

    ymin = max(0, sub["ci_low"].min() - 0.05)
    ymax = min(1, sub["ci_high"].max() + 0.05)
    ax1.set_ylim(ymin, ymax)

    ax2 = ax1.twinx()
    ax2.plot(
        x,
        sub["n_at_risk"],
        linestyle="--",
        alpha=0.6,
        label="n at risk"
    )
    ax2.set_ylabel("n at risk")

    if title is None:
        title = f"{metric}: threshold sweep over {x_col}"

    ax1.set_title(title)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_threshold_sweep(
    threshold_results,
    metric="vu_trap",
    x_col="T_mean",
    direction="above",
    title="P(VU→VU) for transitions above T threshold"
)

plot_threshold_sweep(
    threshold_results,
    metric="discomfort_basin_persistence",
    x_col="T_mean",
    direction="above",
    title="Discomfort-basin persistence above T threshold"
)

plot_threshold_sweep(
    threshold_results,
    metric="recovery_from_discomfort_basin",
    x_col="T_mean",
    direction="above",
    title="Recovery from discomfort basin above T threshold"
)

plot_threshold_sweep(
    threshold_results,
    metric="onset_of_discomfort",
    x_col="T_mean",
    direction="above",
    title="Onset of discomfort above T threshold"
)


# fem el boostrap amb T bins

In [ ]:
metrics_to_test = [
    "onset_of_discomfort",
    "discomfort_basin_persistence",
    "recovery_from_discomfort_basin",
]

x_cols_to_test = [
    # "HDX_mean",
    # "HDX_t1",
    # "HDX_delta",
    # "HDX_t",
    "T_t",
    # "T_delta",
    "T_mean",
    "T_t1",
]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def bootstrap_binned_event_curve(
    transitions,
    x_col,
    metric,
    bins="quantile",
    n_bins=10,
    bin_edges=None,
    min_at_risk=20,
    n_boot=1000,
    seed=123,
    cluster_cols=("ID", "walk_id"),
):
    """
    Estimate P(event | X in bin) with trajectory-level bootstrap.

    Parameters
    ----------
    transitions : pd.DataFrame
        Transition dataframe.
    x_col : str
        Variable used for binning, e.g. T_mean, T_t1, HDX_mean, HDX_t1, T_delta, HDX_delta.
    metric : str
        Event metric, e.g. onset_of_discomfort, discomfort_basin_persistence, recovery_from_discomfort_basin.
    bins : {"quantile", "fixed"}
        "quantile" gives bins with similar sample size.
        "fixed" uses user-defined bin_edges.
    n_bins : int
        Number of quantile bins if bins="quantile".
    bin_edges : list or np.array
        Edges if bins="fixed".
    min_at_risk : int
        Minimum observed number of at-risk transitions required to keep a bin.
    n_boot : int
        Number of bootstrap resamples.
    seed : int
        Random seed.
    cluster_cols : tuple
        Columns defining the resampling unit. Usually ("ID", "walk_id").

    Returns
    -------
    pd.DataFrame
        Binned probability curve with observed probability and bootstrap interval.
    """

    rng = np.random.default_rng(seed)

    event_df = make_event_df(transitions, metric)
    event_df = event_df.dropna(subset=[x_col, "event"]).copy()

    if event_df.empty:
        return pd.DataFrame()

    # --------------------------------------------------
    # 1) Define global bins on observed data
    # --------------------------------------------------
    if bins == "quantile":
        try:
            _, edges = pd.qcut(
                event_df[x_col],
                q=n_bins,
                retbins=True,
                duplicates="drop"
            )
        except ValueError:
            return pd.DataFrame()

    elif bins == "fixed":
        if bin_edges is None:
            raise ValueError("If bins='fixed', you must provide bin_edges.")
        edges = np.asarray(bin_edges, dtype=float)

    else:
        raise ValueError("bins must be either 'quantile' or 'fixed'.")

    edges = np.unique(edges)

    if len(edges) < 2:
        return pd.DataFrame()

    event_df["bin_id"] = pd.cut(
        event_df[x_col],
        bins=edges,
        include_lowest=True,
        labels=False
    )

    event_df = event_df.dropna(subset=["bin_id"]).copy()
    event_df["bin_id"] = event_df["bin_id"].astype(int)

    # --------------------------------------------------
    # 2) Observed probability per bin
    # --------------------------------------------------
    observed_rows = []

    for bin_id, sub in event_df.groupby("bin_id", sort=True):
        n = len(sub)
        if n < min_at_risk:
            continue

        k = int(sub["event"].sum())
        p = k / n

        left = edges[bin_id]
        right = edges[bin_id + 1]

        observed_rows.append({
            "metric": metric,
            "x_col": x_col,
            "bins": bins,
            "n_bins_requested": n_bins if bins == "quantile" else np.nan,
            "bin_id": bin_id,
            "bin_left": left,
            "bin_right": right,
            "bin_label": f"[{left:.2f}, {right:.2f}]",
            "x_center": sub[x_col].mean(),
            "x_median": sub[x_col].median(),
            "x_min_obs": sub[x_col].min(),
            "x_max_obs": sub[x_col].max(),
            "n_at_risk": n,
            "events": k,
            "prob_observed": p,
        })

    observed = pd.DataFrame(observed_rows)

    if observed.empty:
        return observed

    valid_bin_ids = observed["bin_id"].to_numpy()

    # --------------------------------------------------
    # 3) Define bootstrap clusters
    # --------------------------------------------------
    cluster_cols = [c for c in cluster_cols if c in event_df.columns]

    if len(cluster_cols) == 0:
        event_df["_cluster_id"] = event_df.index.astype(str)
    else:
        event_df["_cluster_id"] = (
            event_df[cluster_cols].astype(str).agg("||".join, axis=1)
        )

    cluster_ids = event_df["_cluster_id"].drop_duplicates().to_numpy()

    cluster_groups = {
        cid: g.drop(columns=["_cluster_id"])
        for cid, g in event_df.groupby("_cluster_id", sort=False)
    }

    # --------------------------------------------------
    # 4) Bootstrap
    # --------------------------------------------------
    boot_rows = []

    for b in range(n_boot):
        sampled_ids = rng.choice(cluster_ids, size=len(cluster_ids), replace=True)

        boot_df = pd.concat(
            [cluster_groups[cid] for cid in sampled_ids],
            ignore_index=True
        )

        for bin_id in valid_bin_ids:
            sub = boot_df[boot_df["bin_id"] == bin_id]

            n = len(sub)

            if n == 0:
                boot_rows.append({
                    "boot": b,
                    "bin_id": bin_id,
                    "prob_boot": np.nan,
                    "n_boot_at_risk": 0,
                })
                continue

            k = int(sub["event"].sum())

            boot_rows.append({
                "boot": b,
                "bin_id": bin_id,
                "prob_boot": k / n,
                "n_boot_at_risk": n,
            })

    boot = pd.DataFrame(boot_rows)

    summary = (
        boot
        .dropna(subset=["prob_boot"])
        .groupby("bin_id")
        .agg(
            q025=("prob_boot", lambda x: np.quantile(x, 0.025)),
            median=("prob_boot", lambda x: np.quantile(x, 0.500)),
            q975=("prob_boot", lambda x: np.quantile(x, 0.975)),
            boot_valid=("prob_boot", "count"),
            boot_n_median=("n_boot_at_risk", "median"),
        )
        .reset_index()
    )

    out = observed.merge(summary, on="bin_id", how="left")
    out = out.sort_values("x_center").reset_index(drop=True)

    return out

In [ ]:
binned_results_q10 = []

for metric in metrics_to_test:
    for x_col in x_cols_to_test:
        if x_col in transitions.columns:
            tmp = bootstrap_binned_event_curve(
                transitions=transitions,
                x_col=x_col,
                metric=metric,
                bins="quantile",
                n_bins=10,
                min_at_risk=20,
                n_boot=1000,
                seed=123,
                cluster_cols=("ID", "walk_id"),
            )
            binned_results_q10.append(tmp)

binned_results_q10 = pd.concat(binned_results_q10, ignore_index=True)

binned_results_q10.to_csv(
    "markov_newT/binned_event_probabilities_quantile10.csv",
    index=False
)

# binned_results_q10.head()
binned_results_q10 = pd.read_csv(
    "markov_newT/binned_event_probabilities_quantile10.csv"
)


T_edges = [22, 24, 26, 28, 29, 30, 31, 32, 34]

# binned_T_fixed = []

# for metric in metrics_to_test:
#     for x_col in ["T_mean", "T_t1"]:
#         if x_col in transitions.columns:
#             tmp = bootstrap_binned_event_curve(
#                 transitions=transitions,
#                 x_col=x_col,
#                 metric=metric,
#                 bins="fixed",
#                 bin_edges=T_edges,
#                 min_at_risk=20,
#                 n_boot=1000,
#                 seed=123,
#                 cluster_cols=("ID", "walk_id"),
#             )
#             binned_T_fixed.append(tmp)

# binned_T_fixed = pd.concat(binned_T_fixed, ignore_index=True)

# binned_T_fixed.to_csv(
#     "markov_results/binned_event_probabilities_T_fixed.csv",
#     index=False
# )

# binned_T_fixed.head()

# binned_T_fixed = pd.read_csv(
#     "markov_newT/binned_event_probabilities_T_fixed.csv"
# )

In [ ]:
def plot_binned_event_curve(
    binned_df,
    metric,
    x_col,
    title=None,
    show_n=True,
):
    sub = binned_df[
        (binned_df["metric"] == metric) &
        (binned_df["x_col"] == x_col)
    ].copy()

    if sub.empty:
        print(f"No data for {metric} / {x_col}")
        return

    sub = sub.sort_values("x_center")

    x = sub["x_center"].to_numpy()
    y = sub["prob_observed"].to_numpy()

    fig, ax = plt.subplots(figsize=(7.5, 4.8))

    ax.plot(
        x,
        y,
        marker="o",
        linewidth=1.8,
        label="Observed probability"
    )

    ax.fill_between(
        x,
        sub["q025"],
        sub["q975"],
        alpha=0.22,
        label="Trajectory bootstrap 95% interval"
    )

    if show_n:
        for _, row in sub.iterrows():
            ax.text(
                row["x_center"],
                row["prob_observed"] + 0.015,
                f"n={int(row['n_at_risk'])}",
                ha="center",
                fontsize=8
            )

    ax.set_xlabel(x_col)
    ax.set_ylabel("Event probability")

    if title is None:
        title = f"{metric} by {x_col} bins"

    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()

    ymin = max(0, sub["q025"].min() - 0.05)
    ymax = min(1, sub["q975"].max() + 0.05)
    ax.set_ylim(ymin, ymax)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_binned_event_curve(
    binned_results_q10,
    metric="onset_of_discomfort",
    x_col="T_t1",
    title="Onset probability by T at next stop bins"
)

plot_binned_event_curve(
    binned_results_q10,
    metric="discomfort_basin_persistence",
    x_col="T_t1",
    title="Discomfort-basin persistence by T at next stop bins"
)

plot_binned_event_curve(
    binned_results_q10,
    metric="recovery_from_discomfort_basin",
    x_col="T_t1",
    title="Recovery probability by T at next stop bins"
)

## comparem amb diferentes bins

In [ ]:
def run_bin_sensitivity(
    transitions,
    x_col,
    metric,
    n_bins_list=(6, 8, 10, 12),
    min_at_risk=20,
    n_boot=500,
):
    all_results = []

    for n_bins in n_bins_list:
        tmp = bootstrap_binned_event_curve(
            transitions=transitions,
            x_col=x_col,
            metric=metric,
            bins="quantile",
            n_bins=n_bins,
            min_at_risk=min_at_risk,
            n_boot=n_boot,
            seed=123,
            cluster_cols=("ID", "walk_id"),
        )

        tmp["n_bins"] = n_bins
        all_results.append(tmp)

    return pd.concat(all_results, ignore_index=True)

In [ ]:
# bin_sensitivity_Tt1_basin = run_bin_sensitivity(
#     transitions=transitions,
#     x_col="T_t1",
#     metric="discomfort_basin_persistence",
#     n_bins_list=(6, 8, 10, 12),
#     min_at_risk=20,
#     n_boot=500,
# )

# bin_sensitivity_Tt1_basin.head()
# bin_sensitivity_Tt1_basin.to_csv(
#     "markov_results/bin_sensitivity_Tt1_discomfort_basin_persistence.csv",
#     index=False
# )

bin_sensitivity_Tt1_basin = pd.read_csv(
    "markov_results/bin_sensitivity_Tt1_discomfort_basin_persistence.csv"
)

In [ ]:
def plot_bin_sensitivity(sensitivity_df, title=None):
    fig, ax = plt.subplots(figsize=(8, 5))

    for n_bins, sub in sensitivity_df.groupby("n_bins"):
        sub = sub.sort_values("x_center")

        ax.plot(
            sub["x_center"],
            sub["prob_observed"],
            marker="o",
            linewidth=1.5,
            label=f"{n_bins} bins"
        )

    ax.set_xlabel(sensitivity_df["x_col"].iloc[0])
    ax.set_ylabel("Event probability")

    if title is None:
        title = f"{sensitivity_df['metric'].iloc[0]} bin sensitivity"

    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_bin_sensitivity(
    bin_sensitivity_Tt1_basin,
    title="Bin sensitivity: discomfort-basin persistence vs T at next stop"
)

# probem fent que l'error vingui del boostrap

In [ ]:
def observed_threshold_curve(
    transitions,
    x_col,
    metric,
    direction="above",
    n_grid=60,
    min_at_risk=20,
    q_low=0.05,
    q_high=0.95,
):
    event_df = make_event_df(transitions, metric)
    event_df = event_df.dropna(subset=[x_col, "event"]).copy()

    if event_df.empty:
        return pd.DataFrame(), np.array([])

    thresholds = np.quantile(
        event_df[x_col].to_numpy(),
        np.linspace(q_low, q_high, n_grid)
    )

    thresholds = np.unique(thresholds)

    rows = []

    for thr in thresholds:
        if direction == "above":
            sub = event_df[event_df[x_col] >= thr]
        elif direction == "below":
            sub = event_df[event_df[x_col] <= thr]
        else:
            raise ValueError("direction must be 'above' or 'below'")

        n = len(sub)

        if n < min_at_risk:
            continue

        k = int(sub["event"].sum())
        p = k / n

        rows.append({
            "metric": metric,
            "x_col": x_col,
            "direction": direction,
            "threshold": thr,
            "n_at_risk": n,
            "events": k,
            "prob_observed": p,
        })

    return pd.DataFrame(rows), thresholds

In [ ]:
def bootstrap_threshold_sweep(
    transitions,
    x_col,
    metric,
    direction="above",
    n_grid=60,
    min_at_risk=20,
    q_low=0.05,
    q_high=0.95,
    n_boot=1000,
    seed=123,
    cluster_cols=("ID", "walk_id"),
):
    rng = np.random.default_rng(seed)

    event_df = make_event_df(transitions, metric)
    event_df = event_df.dropna(subset=[x_col, "event"]).copy()

    if event_df.empty:
        return pd.DataFrame()

    # trajectory / participant-walk cluster id
    cluster_cols = [c for c in cluster_cols if c in event_df.columns]

    if len(cluster_cols) == 0:
        event_df["_cluster_id"] = event_df.index.astype(str)
    else:
        event_df["_cluster_id"] = (
            event_df[cluster_cols].astype(str).agg("||".join, axis=1)
        )

    # Use fixed thresholds from observed data
    thresholds = np.quantile(
        event_df[x_col].to_numpy(),
        np.linspace(q_low, q_high, n_grid)
    )
    thresholds = np.unique(thresholds)

    # Observed curve
    observed_rows = []

    for thr in thresholds:
        if direction == "above":
            sub = event_df[event_df[x_col] >= thr]
        elif direction == "below":
            sub = event_df[event_df[x_col] <= thr]
        else:
            raise ValueError("direction must be 'above' or 'below'")

        n = len(sub)

        if n < min_at_risk:
            continue

        k = int(sub["event"].sum())
        observed_rows.append({
            "threshold": thr,
            "n_at_risk": n,
            "events": k,
            "prob_observed": k / n,
        })

    observed = pd.DataFrame(observed_rows)

    if observed.empty:
        return pd.DataFrame()

    valid_thresholds = observed["threshold"].to_numpy()

    # Pre-split by cluster for faster bootstrap
    cluster_ids = event_df["_cluster_id"].drop_duplicates().to_numpy()
    cluster_groups = {
        cid: g.drop(columns=["_cluster_id"])
        for cid, g in event_df.groupby("_cluster_id", sort=False)
    }

    boot_rows = []

    for b in range(n_boot):
        sampled_ids = rng.choice(cluster_ids, size=len(cluster_ids), replace=True)

        boot_df = pd.concat(
            [cluster_groups[cid] for cid in sampled_ids],
            ignore_index=True
        )

        for thr in valid_thresholds:
            if direction == "above":
                sub = boot_df[boot_df[x_col] >= thr]
            else:
                sub = boot_df[boot_df[x_col] <= thr]

            n = len(sub)

            if n < min_at_risk:
                boot_rows.append({
                    "boot": b,
                    "threshold": thr,
                    "prob_boot": np.nan,
                    "n_boot_at_risk": n,
                })
                continue

            k = int(sub["event"].sum())
            boot_rows.append({
                "boot": b,
                "threshold": thr,
                "prob_boot": k / n,
                "n_boot_at_risk": n,
            })

    boot = pd.DataFrame(boot_rows)

    summary = (
        boot
        .dropna(subset=["prob_boot"])
        .groupby("threshold")
        .agg(
            q025=("prob_boot", lambda x: np.quantile(x, 0.025)),
            median=("prob_boot", lambda x: np.quantile(x, 0.50)),
            q975=("prob_boot", lambda x: np.quantile(x, 0.975)),
            boot_valid=("prob_boot", "count"),
            boot_n_median=("n_boot_at_risk", "median"),
        )
        .reset_index()
    )

    out = observed.merge(summary, on="threshold", how="left")

    out["metric"] = metric
    out["x_col"] = x_col
    out["direction"] = direction
    out["condition"] = out.apply(
        lambda r: f"{x_col} >= {r['threshold']:.3f}"
        if direction == "above"
        else f"{x_col} <= {r['threshold']:.3f}",
        axis=1
    )

    # Useful ranking criterion
    out["score_boot_q025"] = out["q025"]

    out = out.sort_values(
        ["score_boot_q025", "prob_observed", "n_at_risk"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return out

In [ ]:
metrics_to_test = [
    "onset_of_discomfort",
    "discomfort_basin_persistence",
    "recovery_from_discomfort_basin",
]

x_cols_to_test = [
    # "HDX_mean",
    # "HDX_t1",
    # "HDX_delta",
    # "HDX_t",
    "T_t",
    # "T_delta",
    "T_mean",
    "T_t1",
]

In [ ]:
bootstrap_threshold_results = []

for metric in metrics_to_test:
    for x_col in x_cols_to_test:
        if x_col in transitions.columns:
            tmp = bootstrap_threshold_sweep(
                transitions=transitions,
                x_col=x_col,
                metric=metric,
                direction="above",
                n_grid=60,
                min_at_risk=20,
                q_low=0.05,
                q_high=0.95,
                n_boot=1000,
                seed=123,
                cluster_cols=("ID", "walk_id"),
            )
            bootstrap_threshold_results.append(tmp)

bootstrap_threshold_results = pd.concat(
    bootstrap_threshold_results,
    ignore_index=True
)

bootstrap_threshold_results.to_csv(
    "markov_newT/bootstrap_threshold_sweep_T_HDX_above.csv",
    index=False
)

bootstrap_threshold_results.head(20)

bootstrap_threshold_results = pd.read_csv("markov_newT/bootstrap_threshold_sweep_T_HDX_above.csv")
# bootstrap_threshold_results_below = pd.read_csv("markov_newT/bootstrap_threshold_sweep_T_HDX_below.csv") 


In [ ]:
def plot_bootstrap_threshold_sweep(
    sweep_df,
    metric,
    x_col,
    direction="above",
    title=None,
):
    sub = sweep_df[
        (sweep_df["metric"] == metric) &
        (sweep_df["x_col"] == x_col) &
        (sweep_df["direction"] == direction)
    ].copy()

    if sub.empty:
        print(f"No results for {metric} / {x_col}")
        return

    sub = sub.sort_values("threshold")

    fig, ax1 = plt.subplots(figsize=(8, 5))

    ax1.plot(
        sub["threshold"],
        sub["prob_observed"],
        "o-",
        label="observed probability"
    )

    ax1.fill_between(
        sub["threshold"],
        sub["q025"],
        sub["q975"],
        alpha=0.25,
        label="trajectory bootstrap 95% interval"
    )

    ax1.set_xlabel(f"{x_col} threshold")
    ax1.set_ylabel(f"P({metric})")
    ax1.grid(alpha=0.3)

    ymin = max(0, np.nanmin(sub["q025"]) - 0.05)
    ymax = min(1, np.nanmax(sub["q975"]) + 0.05)
    ax1.set_ylim(ymin, ymax)

    ax2 = ax1.twinx()
    ax2.plot(
        sub["threshold"],
        sub["n_at_risk"],
        "--",
        alpha=0.6,
        label="n at risk"
    )
    ax2.set_ylabel("n at risk")

    if title is None:
        title = f"{metric}: threshold sweep over {x_col}"

    ax1.set_title(title)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_bootstrap_threshold_sweep(
    bootstrap_threshold_results,
    metric="onset_of_discomfort",
    x_col="T_mean",
    direction="above",
    title="P(onset of discomfort) above T threshold"
)

plot_bootstrap_threshold_sweep(
    bootstrap_threshold_results,
    metric="discomfort_basin_persistence",
    x_col="T_mean",
    direction="above",
    title="Discomfort-basin persistence above T threshold"
)

plot_bootstrap_threshold_sweep(
    bootstrap_threshold_results,
    metric="recovery_from_discomfort_basin",
    x_col="T_mean",
    direction="above",
    title="Recovery from discomfort basin above T threshold"
)

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# # =========================================================
# # CONFIG
# # =========================================================

# hdx_cols = ["HDX_mean", "HDX_t1", "HDX_delta", "HDX_t"]
# t_cols   = ["T_mean", "T_t1", "T_delta", "T_t"]

# metric_title_map = {
#     "onset_of_discomfort": "Onset of discomfort",
#     "recovery_from_discomfort_basin": "Recovery from discomfort basin",
#     "discomfort_basin_persistence": "Discomfort-basin persistence",
# }

# x_title_map = {
#     "HDX_mean": "HDX mean",
#     "HDX_t1": "HDX at next stop",
#     "HDX_delta": "ΔHDX (next - current)",
#     "HDX_t": "HDX at current stop",
#      "T_mean": "T mean",
#     "T_t1": "T at next stop",
#     "T_delta": "ΔT (next - current)",
#     "T_t": "T at current stop",
# }

# # Reference vertical lines (optional)
# vline_map = {
#     "T_mean": 29.0,
#     "T_t1": 29.0,
#     # You can add HDX references if you want:
#     # "HDX_mean": 37.0,
#     # "HDX_t1": 37.0,
# }

# # =========================================================
# # HELPER
# # =========================================================

# def _get_metric_subset(df, metric, x_col, direction="above"):
#     sub = df[
#         (df["metric"] == metric) &
#         (df["x_col"] == x_col) &
#         (df["direction"] == direction)
#     ].copy()

#     if sub.empty:
#         return sub

#     sub = sub.sort_values("threshold").reset_index(drop=True)
#     return sub


# def _compute_global_ylim(df, metric, x_cols, direction="above", pad=0.03):
#     vals_low = []
#     vals_high = []

#     for x_col in x_cols:
#         sub = _get_metric_subset(df, metric, x_col, direction=direction)
#         if sub.empty:
#             continue

#         vals_low.extend(sub["q025"].dropna().tolist())
#         vals_high.extend(sub["q975"].dropna().tolist())

#     if len(vals_low) == 0 or len(vals_high) == 0:
#         return (0, 1)

#     ymin = max(0, min(vals_low) - pad)
#     ymax = min(1, max(vals_high) + pad)

#     if ymax <= ymin:
#         ymin, ymax = 0, 1

#     return ymin, ymax


# def plot_threshold_mosaic(
#     results_df,
#     metric,
#     x_cols,
#     direction="above",
#     figure_title=None,
#     save_path=None,
#     n_at_risk_color="tab:blue",
#     curve_color="tab:blue",
#     band_alpha=0.22,
#     add_reference_lines=True,
#     same_ylim=True,
#     figsize=(12, 8),
# ):
#     """
#     Create a 2x2 mosaic for a given metric across 4 x variables.
#     Expected columns in results_df:
#         metric, x_col, direction, threshold, prob_observed, q025, q975, n_at_risk
#     """

#     if len(x_cols) != 4:
#         raise ValueError("x_cols must contain exactly 4 variables for a 2x2 mosaic.")

#     if figure_title is None:
#         figure_title = metric_title_map.get(metric, metric)

#     if same_ylim:
#         global_ylim = _compute_global_ylim(results_df, metric, x_cols, direction=direction)
#     else:
#         global_ylim = None

#     fig, axes = plt.subplots(2, 2, figsize=figsize, sharey=same_ylim)
#     axes = axes.flatten()

#     panel_labels = ["A", "B", "C", "D"]

#     for i, (ax, x_col) in enumerate(zip(axes, x_cols)):
#         sub = _get_metric_subset(results_df, metric, x_col, direction=direction)

#         if sub.empty:
#             ax.text(0.5, 0.5, f"No data for\n{x_col}", ha="center", va="center", fontsize=11)
#             ax.set_axis_off()
#             continue

#         # Main curve
#         ax.plot(
#             sub["threshold"],
#             sub["prob_observed"],
#             marker="o",
#             markersize=4,
#             linewidth=1.8,
#             color=curve_color,
#             label="Observed probability",
#             zorder=3
#         )

#         # Bootstrap band
#         ax.fill_between(
#             sub["threshold"],
#             sub["q025"],
#             sub["q975"],
#             color=curve_color,
#             alpha=band_alpha,
#             label="Trajectory bootstrap 95% interval",
#             zorder=2
#         )

#         # Secondary axis for n_at_risk
#         ax2 = ax.twinx()
#         ax2.plot(
#             sub["threshold"],
#             sub["n_at_risk"],
#             linestyle="--",
#             linewidth=1.4,
#             color=n_at_risk_color,
#             alpha=0.75,
#             label="n at risk",
#             zorder=1
#         )

#         # Formatting
#         ax.set_title(f"{panel_labels[i]}. {x_title_map.get(x_col, x_col)}", fontsize=11)
#         ax.set_xlabel("Threshold")
#         if i in [0, 2]:
#             ax.set_ylabel("Event probability")
#         if i in [1, 3]:
#             ax2.set_ylabel("n at risk")

#         ax.grid(alpha=0.25)

#         if same_ylim and global_ylim is not None:
#             ax.set_ylim(*global_ylim)
#         else:
#             local_ymin = max(0, sub["q025"].min() - 0.03)
#             local_ymax = min(1, sub["q975"].max() + 0.03)
#             ax.set_ylim(local_ymin, local_ymax)

#         # Optional vertical reference line
#         if add_reference_lines and x_col in vline_map:
#             ax.axvline(
#                 vline_map[x_col],
#                 color="black",
#                 linestyle=":",
#                 linewidth=1.2,
#                 alpha=0.8
#             )

#         # Make right axis lighter
#         ax2.tick_params(axis="y", labelsize=9, colors="dimgray")
#         ax2.spines["right"].set_alpha(0.5)

#     # Global title
#     fig.suptitle(figure_title, fontsize=14, y=0.98)

#     # Shared legend
#     from matplotlib.lines import Line2D
#     from matplotlib.patches import Patch

#     legend_handles = [
#         Line2D([0], [0], color=curve_color, marker="o", linewidth=1.8, markersize=5, label="Observed probability"),
#         Patch(facecolor=curve_color, alpha=band_alpha, label="Trajectory bootstrap 95% interval"),
#         Line2D([0], [0], color=n_at_risk_color, linestyle="--", linewidth=1.4, label="n at risk"),
#     ]

#     fig.legend(
#         handles=legend_handles,
#         loc="lower center",
#         ncol=3,
#         frameon=False,
#         bbox_to_anchor=(0.5, -0.01)
#     )

#     plt.tight_layout(rect=[0, 0.04, 1, 0.95])

#     if save_path is not None:
#         plt.savefig(save_path, dpi=300, bbox_inches="tight")

#     plt.show()

In [ ]:
# # 1) Onset — HDX
# plot_threshold_mosaic(
#     results_df=bootstrap_threshold_results,
#     metric="onset_of_discomfort",
#     x_cols=hdx_cols,
#     direction="above",
#     figure_title="Onset of discomfort — HDX thresholds",
#     save_path="paper_fig_onset_HDX_mosaic.png"
# )

# # 2) Onset — T
# plot_threshold_mosaic(
#     results_df=bootstrap_threshold_results,
#     metric="onset_of_discomfort",
#     x_cols=t_cols,
#     direction="above",
#     figure_title="Onset of discomfort — Temperature thresholds",
#     save_path="paper_fig_onset_T_mosaic.png"
# )

# # 3) Recovery — HDX
# plot_threshold_mosaic(
#     results_df=bootstrap_threshold_results,
#     metric="recovery_from_discomfort_basin",
#     x_cols=hdx_cols,
#     direction="above",
#     figure_title="Recovery from discomfort basin — HDX thresholds",
#     save_path="paper_fig_recovery_HDX_mosaic.png"
# )

# # 4) Recovery — T
# plot_threshold_mosaic(
#     results_df=bootstrap_threshold_results,
#     metric="recovery_from_discomfort_basin",
#     x_cols=t_cols,
#     direction="above",
#     figure_title="Recovery from discomfort basin — Temperature thresholds",
#     save_path="paper_fig_recovery_T_mosaic.png"
# )

# # 5) Basin — HDX
# plot_threshold_mosaic(
#     results_df=bootstrap_threshold_results,
#     metric="discomfort_basin_persistence",
#     x_cols=hdx_cols,
#     direction="above",
#     figure_title="Discomfort-basin persistence — HDX thresholds",
#     save_path="paper_fig_basin_HDX_mosaic.png"
# )

# # 6) Basin — T
# plot_threshold_mosaic(
#     results_df=bootstrap_threshold_results,
#     metric="discomfort_basin_persistence",
#     x_cols=t_cols,
#     direction="above",
#     figure_title="Discomfort-basin persistence — Temperature thresholds",
#     save_path="paper_fig_basin_T_mosaic.png"
# )

# Figures boniques Per a les P(onset), P(recovery) i P(persistance)

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# strong_x_cols = ["HDX_mean", "HDX_t1", "T_mean", "T_t1"]

# metric_order = [
#     "onset_of_discomfort",
#     "discomfort_basin_persistence",
#     "recovery_from_discomfort_basin",
# ]

# metric_title_map = {
#     "onset_of_discomfort": "Onset of discomfort",
#     "discomfort_basin_persistence": "Discomfort-basin persistence",
#     "recovery_from_discomfort_basin": "Recovery from discomfort basin",
# }

# x_title_map = {
#     "HDX_mean": "HDX mean",
#     "HDX_t1": "HDX at next stop",
#     "T_mean": "Temperature mean",
#     "T_t1": "Temperature at next stop",
# }

# # si vols marcar el possible canvi de règim
# reference_line_map = {
#     "T_mean": 29.0,
#     "T_t1": 29.0,
#     # si algun dia vols posar HDX:
#     # "HDX_mean": 37.0,
#     # "HDX_t1": 37.0,
# }

In [ ]:
# def get_subset(results_df, metric, x_col, direction="above"):
#     sub = results_df[
#         (results_df["metric"] == metric) &
#         (results_df["x_col"] == x_col) &
#         (results_df["direction"] == direction)
#     ].copy()

#     if sub.empty:
#         return sub

#     return sub.sort_values("threshold").reset_index(drop=True)


# def compute_metric_ylim(results_df, metric, x_cols, direction="above", pad=0.03):
#     lows, highs = [], []

#     for x_col in x_cols:
#         sub = get_subset(results_df, metric, x_col, direction=direction)
#         if sub.empty:
#             continue
#         lows.extend(sub["q025"].dropna().tolist())
#         highs.extend(sub["q975"].dropna().tolist())

#     if len(lows) == 0:
#         return (0, 1)

#     ymin = max(0, min(lows) - pad)
#     ymax = min(1, max(highs) + pad)

#     if ymax <= ymin:
#         return (0, 1)

#     return ymin, ymax

In [ ]:
# def plot_metric_mosaic_2x2(
#     results_df,
#     metric,
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=True,
#     reference_line_map=None,
#     use_reference_band=False,
#     band_halfwidth=0.15,
#     figsize=(11, 8),
#     save_path=None,
# ):
#     if len(x_cols) != 4:
#         raise ValueError("x_cols must have exactly 4 elements for a 2x2 mosaic.")

#     fig, axes = plt.subplots(2, 2, figsize=figsize, sharey=True)
#     axes = axes.flatten()

#     panel_labels = ["A", "B", "C", "D"]
#     ylims = compute_metric_ylim(results_df, metric, x_cols, direction=direction)

#     for i, (ax, x_col) in enumerate(zip(axes, x_cols)):
#         sub = get_subset(results_df, metric, x_col, direction=direction)

#         if sub.empty:
#             ax.text(0.5, 0.5, f"No data for {x_col}", ha="center", va="center")
#             ax.set_axis_off()
#             continue

#         # main observed curve
#         ax.plot(
#             sub["threshold"],
#             sub["prob_observed"],
#             color="tab:blue",
#             marker="o",
#             markersize=4,
#             linewidth=1.8,
#             zorder=3
#         )

#         # bootstrap band
#         ax.fill_between(
#             sub["threshold"],
#             sub["q025"],
#             sub["q975"],
#             color="tab:blue",
#             alpha=0.22,
#             zorder=2
#         )

#         # optional reference line / band
#         if add_reference_lines and reference_line_map is not None and x_col in reference_line_map:
#             ref = reference_line_map[x_col]

#             if use_reference_band:
#                 ax.axvspan(
#                     ref - band_halfwidth,
#                     ref + band_halfwidth,
#                     color="gray",
#                     alpha=0.10,
#                     zorder=1
#                 )
#             else:
#                 ax.axvline(
#                     ref,
#                     color="black",
#                     linestyle=":",
#                     linewidth=1.2,
#                     alpha=0.8,
#                     zorder=1
#                 )

#         # n at risk on twin axis
#         ax2 = ax.twinx()
#         ax2.plot(
#             sub["threshold"],
#             sub["n_at_risk"],
#             linestyle="--",
#             color="tab:blue",
#             alpha=0.65,
#             linewidth=1.4
#         )

#         # titles / labels
#         ax.set_title(f"{panel_labels[i]}. {x_title_map.get(x_col, x_col)}", fontsize=11)
#         ax.set_xlabel("Threshold")
#         if i in [0, 2]:
#             ax.set_ylabel("Event probability")
#         if i in [1, 3]:
#             ax2.set_ylabel("n at risk")

#         ax.grid(alpha=0.25)
#         ax.set_ylim(*ylims)

#         ax2.tick_params(axis="y", labelsize=9, colors="dimgray")
#         ax2.spines["right"].set_alpha(0.5)

#     fig.suptitle(metric_title_map.get(metric, metric), fontsize=14, y=0.98)

#     from matplotlib.lines import Line2D
#     from matplotlib.patches import Patch

#     legend_handles = [
#         Line2D([0], [0], color="tab:blue", marker="o", linewidth=1.8, markersize=5,
#                label="Observed probability"),
#         Patch(facecolor="tab:blue", alpha=0.22, label="Trajectory bootstrap 95% interval"),
#         Line2D([0], [0], color="tab:blue", linestyle="--", linewidth=1.4,
#                label="n at risk"),
#     ]

#     if add_reference_lines and reference_line_map is not None:
#         if use_reference_band:
#             legend_handles.append(
#                 Patch(facecolor="gray", alpha=0.10, label="Reference regime")
#             )
#         else:
#             legend_handles.append(
#                 Line2D([0], [0], color="black", linestyle=":", linewidth=1.2,
#                        label="Reference threshold")
#             )

#     fig.legend(
#         handles=legend_handles,
#         loc="lower center",
#         ncol=4,
#         frameon=False,
#         bbox_to_anchor=(0.5, -0.01)
#     )

#     plt.tight_layout(rect=[0, 0.05, 1, 0.95])

#     if save_path is not None:
#         plt.savefig(save_path, dpi=300, bbox_inches="tight")

#     plt.show()

In [ ]:
# def plot_summary_3x4(
#     results_df,
#     metrics=metric_order,
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=True,
#     reference_line_map=None,
#     use_reference_band=False,
#     band_halfwidth=0.15,
#     figsize=(15, 10),
#     save_path=None,
# ):
#     nrows = len(metrics)
#     ncols = len(x_cols)

#     fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=False, sharey="row")

#     if nrows == 1:
#         axes = np.array([axes])

#     # y-limits by row (metric)
#     metric_ylims = {
#         metric: compute_metric_ylim(results_df, metric, x_cols, direction=direction)
#         for metric in metrics
#     }

#     panel_letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
#     panel_counter = 0

#     for r, metric in enumerate(metrics):
#         for c, x_col in enumerate(x_cols):
#             ax = axes[r, c]
#             sub = get_subset(results_df, metric, x_col, direction=direction)

#             if sub.empty:
#                 ax.text(0.5, 0.5, f"No data", ha="center", va="center")
#                 ax.set_axis_off()
#                 continue

#             # observed probability
#             ax.plot(
#                 sub["threshold"],
#                 sub["prob_observed"],
#                 color="tab:blue",
#                 marker="o",
#                 markersize=3.8,
#                 linewidth=1.6,
#                 zorder=3
#             )

#             # bootstrap band
#             ax.fill_between(
#                 sub["threshold"],
#                 sub["q025"],
#                 sub["q975"],
#                 color="tab:blue",
#                 alpha=0.20,
#                 zorder=2
#             )

#             # reference line / band
#             if add_reference_lines and reference_line_map is not None and x_col in reference_line_map:
#                 ref = reference_line_map[x_col]

#                 if use_reference_band:
#                     ax.axvspan(
#                         ref - band_halfwidth,
#                         ref + band_halfwidth,
#                         color="gray",
#                         alpha=0.10,
#                         zorder=1
#                     )
#                 else:
#                     ax.axvline(
#                         ref,
#                         color="black",
#                         linestyle=":",
#                         linewidth=1.1,
#                         alpha=0.8,
#                         zorder=1
#                     )

#             # twin axis for n at risk
#             ax2 = ax.twinx()
#             ax2.plot(
#                 sub["threshold"],
#                 sub["n_at_risk"],
#                 linestyle="--",
#                 color="tab:blue",
#                 alpha=0.60,
#                 linewidth=1.2,
#                 zorder=1
#             )

#             ax.set_ylim(*metric_ylims[metric])
#             ax.grid(alpha=0.22)

#             # titles only first row
#             if r == 0:
#                 ax.set_title(x_title_map.get(x_col, x_col), fontsize=11)

#             # y-label only first col
#             if c == 0:
#                 ax.set_ylabel(metric_title_map.get(metric, metric), fontsize=10)

#             # x-label only bottom row
#             if r == nrows - 1:
#                 ax.set_xlabel("Threshold")

#             # only last col shows right-axis label
#             if c == ncols - 1:
#                 ax2.set_ylabel("n at risk", fontsize=9)
#             else:
#                 ax2.set_yticklabels([])

#             ax2.tick_params(axis="y", labelsize=8, colors="dimgray")
#             ax2.spines["right"].set_alpha(0.45)

#             # panel label
#             ax.text(
#                 0.02, 0.96,
#                 panel_letters[panel_counter],
#                 transform=ax.transAxes,
#                 ha="left", va="top",
#                 fontsize=11, fontweight="bold"
#             )
#             panel_counter += 1

#     from matplotlib.lines import Line2D
#     from matplotlib.patches import Patch

#     legend_handles = [
#         Line2D([0], [0], color="tab:blue", marker="o", linewidth=1.6, markersize=4,
#                label="Observed probability"),
#         Patch(facecolor="tab:blue", alpha=0.20, label="Trajectory bootstrap 95% interval"),
#         Line2D([0], [0], color="tab:blue", linestyle="--", linewidth=1.2,
#                label="n at risk"),
#     ]

#     if add_reference_lines and reference_line_map is not None:
#         if use_reference_band:
#             legend_handles.append(
#                 Patch(facecolor="gray", alpha=0.10, label="Reference regime")
#             )
#         else:
#             legend_handles.append(
#                 Line2D([0], [0], color="black", linestyle=":", linewidth=1.1,
#                        label="Reference threshold")
#             )

#     fig.legend(
#         handles=legend_handles,
#         loc="lower center",
#         ncol=4,
#         frameon=False,
#         bbox_to_anchor=(0.5, -0.01)
#     )

#     plt.tight_layout(rect=[0, 0.05, 1, 0.98])

#     if save_path is not None:
#         plt.savefig(save_path, dpi=300, bbox_inches="tight")

#     plt.show()

## Generem els plots

In [ ]:
# plot_summary_3x4(
#     results_df=bootstrap_threshold_results,
#     metrics=metric_order,
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=False,
#     reference_line_map=reference_line_map,
#     use_reference_band=False,   # posa True si prefereixes banda suau
#     figsize=(15, 10),
#     save_path="paper_summary_3x4_strong_variables.png"
# )

In [ ]:
# plot_summary_3x4(
#     results_df=bootstrap_threshold_results,
#     metrics=metric_order,
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=True,
#     reference_line_map=reference_line_map,
#     use_reference_band=True,
#     band_halfwidth=0.15,
#     figsize=(15, 10),
#     save_path="paper_summary_3x4_strong_variables_band.png"
# )

In [ ]:
# plot_summary_3x4(
#     results_df=bootstrap_threshold_results,
#     metrics=metric_order,
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=True,
#     reference_line_map=reference_line_map,
#     use_reference_band=True,
#     band_halfwidth=0.15,
#     figsize=(15, 10),
#     save_path="paper_summary_3x4_strong_variables_band.png"
# )

## Mosaics per Probabilitat

In [ ]:
# plot_metric_mosaic_2x2(
#     results_df=bootstrap_threshold_results,
#     metric="onset_of_discomfort",
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=True,
#     reference_line_map=reference_line_map,
#     use_reference_band=True,
#     save_path="paper_mosaic_onset_2x2.png"
# )

In [ ]:
# plot_metric_mosaic_2x2(
#     results_df=bootstrap_threshold_results,
#     metric="discomfort_basin_persistence",
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=True,
#     reference_line_map=reference_line_map,
#     use_reference_band=True,
#     save_path="paper_mosaic_basin_2x2.png"
# )

In [ ]:
# plot_metric_mosaic_2x2(
#     results_df=bootstrap_threshold_results,
#     metric="recovery_from_discomfort_basin",
#     x_cols=strong_x_cols,
#     direction="above",
#     add_reference_lines=True,
#     reference_line_map=reference_line_map,
#     use_reference_band=True,
#     save_path="paper_mosaic_recovery_2x2.png"
# )

## i si ara ho comprobem només amb els Deltes

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.lines import Line2D
# from matplotlib.patches import Patch

# delta_x_cols = ["HDX_delta", "T_delta"]

# metric_order = [
#     "onset_of_discomfort",
#     "discomfort_basin_persistence",
#     "recovery_from_discomfort_basin",
# ]

# metric_title_map = {
#     "onset_of_discomfort": "Onset of discomfort",
#     "discomfort_basin_persistence": "Discomfort-basin persistence",
#     "recovery_from_discomfort_basin": "Recovery from discomfort basin",
# }

# x_title_map = {
#     "HDX_delta": r"$\Delta$HDX = HDX$_{t+1}$ - HDX$_t$",
#     "T_delta": r"$\Delta$T = T$_{t+1}$ - T$_t$",
# }

In [ ]:
# def get_subset(results_df, metric, x_col, direction="above"):
#     sub = results_df[
#         (results_df["metric"] == metric) &
#         (results_df["x_col"] == x_col) &
#         (results_df["direction"] == direction)
#     ].copy()

#     if sub.empty:
#         return sub

#     return sub.sort_values("threshold").reset_index(drop=True)


# def compute_metric_ylim(results_df, metric, x_cols, direction="above", pad=0.03):
#     lows, highs = [], []

#     for x_col in x_cols:
#         sub = get_subset(results_df, metric, x_col, direction=direction)

#         if sub.empty:
#             continue

#         lows.extend(sub["q025"].dropna().tolist())
#         highs.extend(sub["q975"].dropna().tolist())

#     if len(lows) == 0:
#         return (0, 1)

#     ymin = max(0, min(lows) - pad)
#     ymax = min(1, max(highs) + pad)

#     if ymax <= ymin:
#         return (0, 1)

#     return ymin, ymax


# def plot_delta_summary_3x2(
#     results_df,
#     metrics=metric_order,
#     x_cols=delta_x_cols,
#     direction="above",
#     figsize=(11, 9),
#     save_path="paper_delta_summary_3x2.png",
# ):
#     nrows = len(metrics)
#     ncols = len(x_cols)

#     fig, axes = plt.subplots(
#         nrows,
#         ncols,
#         figsize=figsize,
#         sharex=False,
#         sharey="row"
#     )

#     if nrows == 1:
#         axes = np.array([axes])

#     metric_ylims = {
#         metric: compute_metric_ylim(
#             results_df,
#             metric,
#             x_cols,
#             direction=direction
#         )
#         for metric in metrics
#     }

#     panel_letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
#     panel_counter = 0

#     for r, metric in enumerate(metrics):
#         for c, x_col in enumerate(x_cols):
#             ax = axes[r, c]

#             sub = get_subset(
#                 results_df,
#                 metric=metric,
#                 x_col=x_col,
#                 direction=direction
#             )

#             if sub.empty:
#                 ax.text(
#                     0.5, 0.5,
#                     f"No data\n{metric}\n{x_col}",
#                     ha="center",
#                     va="center"
#                 )
#                 ax.set_axis_off()
#                 continue

#             # Observed probability
#             ax.plot(
#                 sub["threshold"],
#                 sub["prob_observed"],
#                 color="tab:blue",
#                 marker="o",
#                 markersize=3.8,
#                 linewidth=1.6,
#                 zorder=3
#             )

#             # Bootstrap band
#             ax.fill_between(
#                 sub["threshold"],
#                 sub["q025"],
#                 sub["q975"],
#                 color="tab:blue",
#                 alpha=0.20,
#                 zorder=2
#             )

#             # Reference line: zero change
#             ax.axvline(
#                 0,
#                 color="black",
#                 linestyle=":",
#                 linewidth=1.2,
#                 alpha=0.85,
#                 zorder=1
#             )

#             # n at risk
#             ax2 = ax.twinx()
#             ax2.plot(
#                 sub["threshold"],
#                 sub["n_at_risk"],
#                 linestyle="--",
#                 color="tab:blue",
#                 alpha=0.60,
#                 linewidth=1.2,
#                 zorder=1
#             )

#             ax.set_ylim(*metric_ylims[metric])
#             ax.grid(alpha=0.22)

#             if r == 0:
#                 ax.set_title(x_title_map.get(x_col, x_col), fontsize=12)

#             if c == 0:
#                 ax.set_ylabel(
#                     metric_title_map.get(metric, metric),
#                     fontsize=10
#                 )

#             if r == nrows - 1:
#                 ax.set_xlabel("Threshold")

#             if c == ncols - 1:
#                 ax2.set_ylabel("n at risk", fontsize=9)
#             else:
#                 ax2.set_yticklabels([])

#             ax2.tick_params(axis="y", labelsize=8, colors="dimgray")
#             ax2.spines["right"].set_alpha(0.45)

#             ax.text(
#                 0.02, 0.96,
#                 panel_letters[panel_counter],
#                 transform=ax.transAxes,
#                 ha="left",
#                 va="top",
#                 fontsize=11,
#                 fontweight="bold"
#             )
#             panel_counter += 1

#     legend_handles = [
#         Line2D(
#             [0], [0],
#             color="tab:blue",
#             marker="o",
#             linewidth=1.6,
#             markersize=4,
#             label="Observed probability"
#         ),
#         Patch(
#             facecolor="tab:blue",
#             alpha=0.20,
#             label="Trajectory bootstrap 95% interval"
#         ),
#         Line2D(
#             [0], [0],
#             color="tab:blue",
#             linestyle="--",
#             linewidth=1.2,
#             label="n at risk"
#         ),
#         Line2D(
#             [0], [0],
#             color="black",
#             linestyle=":",
#             linewidth=1.2,
#             label="zero change"
#         ),
#     ]

#     fig.legend(
#         handles=legend_handles,
#         loc="lower center",
#         ncol=4,
#         frameon=False,
#         bbox_to_anchor=(0.5, -0.01)
#     )

#     fig.suptitle(
#         "Transition probabilities above thermal-change thresholds",
#         fontsize=14,
#         y=0.985
#     )

#     plt.tight_layout(rect=[0, 0.05, 1, 0.965])

#     if save_path is not None:
#         plt.savefig(save_path, dpi=300, bbox_inches="tight")

#     plt.show()
    

In [ ]:
plot_delta_summary_3x2(
    results_df=bootstrap_threshold_results,
    metrics=metric_order,
    x_cols=["HDX_delta", "T_delta"],
    direction="above",
    figsize=(11, 9),
    save_path="paper_delta_summary_3x2.png"
)

# Aprofitem per a tirar les coses per a la sensació tèrmica

In [ ]:

# prepared_path = Path("saved_df/df_dynamics_with_oof_sensation.csv")
# df_prepared = pd.read_csv(prepared_path)
# print(df_prepared.shape)
# df_prepared.columns


In [ ]:
# state_order = ["cool-neutral", "warm-hot", "very hot"]
# state_num_map = {s: i for i, s in enumerate(state_order)}


# def build_transitions_for_trap_sweep(
#     df_prepared,
#     state_col = "sens3_option2"  ,
#     t_col = "<T-T_fixed+<T>>",
#     hdx_col = "<HDX-HDX_fixed+<HDX>>",
#     max_stop=5,
# ):
#     df = df_prepared.copy()

#     sort_cols = [c for c in ["ID", "walk_id", "stop_idx", "row_id"] if c in df.columns]
#     group_cols = [c for c in ["ID", "walk_id"] if c in df.columns]

#     df = df.sort_values(sort_cols).reset_index(drop=True)
#     g = df.groupby(group_cols, sort=False)

#     df["state"] = df[state_col]
#     df["state_num"] = df["state"].map(state_num_map)

#     df["next_state"] = g["state"].shift(-1)
#     df["next_state_num"] = g["state_num"].shift(-1)
#     df["next_stop_idx"] = g["stop_idx"].shift(-1)

#     df["step_gap"] = df["next_stop_idx"] - df["stop_idx"]

#     # T variables
#     if t_col in df.columns:
#         df["T_t"] = df[t_col]
#         df["T_t1"] = g[t_col].shift(-1)
#         df["T_mean"] = 0.5 * (df["T_t"] + df["T_t1"])
#         df["T_delta"] = df["T_t1"] - df["T_t"]

#     # HDX variables
#     if hdx_col in df.columns:
#         df["HDX_t"] = df[hdx_col]
#         df["HDX_t1"] = g[hdx_col].shift(-1)
#         df["HDX_mean"] = 0.5 * (df["HDX_t"] + df["HDX_t1"])
#         df["HDX_delta"] = df["HDX_t1"] - df["HDX_t"]

#     transitions = df[
#         (df["step_gap"] == 1) &
#         (df["stop_idx"] <= max_stop) &
#         (df["next_stop_idx"] <= max_stop)
#     ].copy()

#     transitions = transitions.dropna(subset=["state", "next_state"]).copy()
#     transitions["delta_state"] = transitions["next_state_num"] - transitions["state_num"]

#     return transitions


In [ ]:
# transitions = build_transitions_for_trap_sweep(
#     df_prepared,
#     max_stop=5
# )

# transitions.shape

In [ ]:
# # canviar-ho tot en el format de la Sensació



# def make_event_df(transitions, metric):
#     df = transitions.copy()

#     if metric == "vh_trap":
#         # P(VH -> VH)
#         sub = df[df["state"] == "very hot"].copy()
#         sub["event"] = (sub["next_state"] == "very hot").astype(int)

#     # elif metric == "u_persistence":
#     #     # P(U -> U)
#     #     sub = df[df["state"] == "uncomfortable"].copy()
#     #     sub["event"] = (sub["next_state"] == "uncomfortable").astype(int)

#     # elif metric == "u_adverse_persistence":
#     #     # P(U -> U or VU)
#     #     sub = df[df["state"] == "uncomfortable"].copy()
#     #     sub["event"] = sub["next_state"].isin([
#     #         "uncomfortable",
#     #         "very uncomfortable"
#     #     ]).astype(int)

#     elif metric == "warmth_basin_persistence":
#         # P(next in {U,VU} | current in {U,VU})
#         sub = df[df["state"].isin([
#             "warm-hot",
#             "very hot"
#         ])].copy()
#         sub["event"] = sub["next_state"].isin([
#             "warm-hot",
#             "very hot"
#         ]).astype(int)

#     elif metric == "recovery_from_warmth_basin":
#         # P(next = CN | current in {U,VU})
#         sub = df[df["state"].isin([
#             "warm-hot",
#             "very hot"
#         ])].copy()
#         sub["event"] = (sub["next_state"] == "cool-neutral").astype(int)

#     elif metric == "onset_of_warmth":
#         # P(next in {U,VU} | current = CN)
#         sub = df[df["state"] == "cool-neutral"].copy()
#         sub["event"] = sub["next_state"].isin([
#             "warm-hot",
#             "very hot"
#         ]).astype(int)

#     else:
#         raise ValueError(f"Unknown metric: {metric}")

#     return sub

## ara generem el boostroap

In [ ]:
# metrics_to_test = [
#     "vh_trap",
#     "warmth_basin_persistence",
#     "recovery_from_warmth_basin",
#     "onset_of_warmth",
# ]

# x_cols_to_test = [
#     "HDX_mean",
#     "HDX_t1",
#     "HDX_delta",
#     "HDX_t",
#     "T_t",
#     "T_delta",
#     "T_mean",
#     "T_t1",
# ]

In [ ]:
# def bootstrap_threshold_sweep(
#     transitions,
#     x_col,
#     metric,
#     direction="above",
#     n_grid=60,
#     min_at_risk=20,
#     q_low=0.05,
#     q_high=0.95,
#     n_boot=1000,
#     seed=123,
#     cluster_cols=("ID", "walk_id"),
# ):
#     rng = np.random.default_rng(seed)

#     event_df = make_event_df(transitions, metric)
#     event_df = event_df.dropna(subset=[x_col, "event"]).copy()

#     if event_df.empty:
#         return pd.DataFrame()

#     # trajectory / participant-walk cluster id
#     cluster_cols = [c for c in cluster_cols if c in event_df.columns]

#     if len(cluster_cols) == 0:
#         event_df["_cluster_id"] = event_df.index.astype(str)
#     else:
#         event_df["_cluster_id"] = (
#             event_df[cluster_cols].astype(str).agg("||".join, axis=1)
#         )

#     # Use fixed thresholds from observed data
#     thresholds = np.quantile(
#         event_df[x_col].to_numpy(),
#         np.linspace(q_low, q_high, n_grid)
#     )
#     thresholds = np.unique(thresholds)

#     # Observed curve
#     observed_rows = []

#     for thr in thresholds:
#         if direction == "above":
#             sub = event_df[event_df[x_col] >= thr]
#         elif direction == "below":
#             sub = event_df[event_df[x_col] <= thr]
#         else:
#             raise ValueError("direction must be 'above' or 'below'")

#         n = len(sub)

#         if n < min_at_risk:
#             continue

#         k = int(sub["event"].sum())
#         observed_rows.append({
#             "threshold": thr,
#             "n_at_risk": n,
#             "events": k,
#             "prob_observed": k / n,
#         })

#     observed = pd.DataFrame(observed_rows)

#     if observed.empty:
#         return pd.DataFrame()

#     valid_thresholds = observed["threshold"].to_numpy()

#     # Pre-split by cluster for faster bootstrap
#     cluster_ids = event_df["_cluster_id"].drop_duplicates().to_numpy()
#     cluster_groups = {
#         cid: g.drop(columns=["_cluster_id"])
#         for cid, g in event_df.groupby("_cluster_id", sort=False)
#     }

#     boot_rows = []

#     for b in range(n_boot):
#         sampled_ids = rng.choice(cluster_ids, size=len(cluster_ids), replace=True)

#         boot_df = pd.concat(
#             [cluster_groups[cid] for cid in sampled_ids],
#             ignore_index=True
#         )

#         for thr in valid_thresholds:
#             if direction == "above":
#                 sub = boot_df[boot_df[x_col] >= thr]
#             else:
#                 sub = boot_df[boot_df[x_col] <= thr]

#             n = len(sub)

#             if n < min_at_risk:
#                 boot_rows.append({
#                     "boot": b,
#                     "threshold": thr,
#                     "prob_boot": np.nan,
#                     "n_boot_at_risk": n,
#                 })
#                 continue

#             k = int(sub["event"].sum())
#             boot_rows.append({
#                 "boot": b,
#                 "threshold": thr,
#                 "prob_boot": k / n,
#                 "n_boot_at_risk": n,
#             })

#     boot = pd.DataFrame(boot_rows)

#     summary = (
#         boot
#         .dropna(subset=["prob_boot"])
#         .groupby("threshold")
#         .agg(
#             q025=("prob_boot", lambda x: np.quantile(x, 0.025)),
#             median=("prob_boot", lambda x: np.quantile(x, 0.50)),
#             q975=("prob_boot", lambda x: np.quantile(x, 0.975)),
#             boot_valid=("prob_boot", "count"),
#             boot_n_median=("n_boot_at_risk", "median"),
#         )
#         .reset_index()
#     )

#     out = observed.merge(summary, on="threshold", how="left")

#     out["metric"] = metric
#     out["x_col"] = x_col
#     out["direction"] = direction
#     out["condition"] = out.apply(
#         lambda r: f"{x_col} >= {r['threshold']:.3f}"
#         if direction == "above"
#         else f"{x_col} <= {r['threshold']:.3f}",
#         axis=1
#     )

#     # Useful ranking criterion
#     out["score_boot_q025"] = out["q025"]

#     out = out.sort_values(
#         ["score_boot_q025", "prob_observed", "n_at_risk"],
#         ascending=[False, False, False]
#     ).reset_index(drop=True)

#     return out

In [ ]:
# bootstrap_threshold_results = []

# for metric in metrics_to_test:
#     for x_col in x_cols_to_test:
#         if x_col in transitions.columns:
#             tmp = bootstrap_threshold_sweep(
#                 transitions=transitions,
#                 x_col=x_col,
#                 metric=metric,
#                 direction="below",
#                 n_grid=60,
#                 min_at_risk=20,
#                 q_low=0.05,
#                 q_high=0.95,
#                 n_boot=1000,
#                 seed=123,
#                 cluster_cols=("ID", "walk_id"),
#             )
#             bootstrap_threshold_results.append(tmp)

# bootstrap_threshold_results = pd.concat(
#     bootstrap_threshold_results,
#     ignore_index=True
# )

# bootstrap_threshold_results.to_csv(
#     "markov_results/sensation_bootstrap_threshold_sweep_T_HDX_below.csv",
#     index=False
# )


# ============================================================
# ABOVE
# ============================================================

# bootstrap_threshold_results_above_list = []

# for metric in metrics_to_test:
#     for x_col in x_cols_to_test:
#         if x_col in transitions.columns:
#             print(f"Running ABOVE | metric={metric} | x_col={x_col}")

#             tmp = bootstrap_threshold_sweep(
#                 transitions=transitions,
#                 x_col=x_col,
#                 metric=metric,
#                 direction="above",
#                 n_grid=60,
#                 min_at_risk=20,
#                 q_low=0.05,
#                 q_high=0.95,
#                 n_boot=1000,
#                 seed=123,
#                 cluster_cols=("ID", "walk_id"),
#             )

#             if not tmp.empty:
#                 bootstrap_threshold_results_above_list.append(tmp)

# bootstrap_threshold_results_above = pd.concat(
#     bootstrap_threshold_results_above_list,
#     ignore_index=True
# )

# bootstrap_threshold_results_above.to_csv(
#     "markov_results/sensation_bootstrap_threshold_sweep_T_HDX_above.csv",
#     index=False
# )

# bootstrap_threshold_results.head(20)

# bootstrap_threshold_results_above = pd.read_csv("markov_results/sensation_bootstrap_threshold_sweep_T_HDX_above.csv")
# bootstrap_threshold_results_below = pd.read_csv("markov_results/sensation_bootstrap_threshold_sweep_T_HDX_below.csv") 


# Ara binari

In [ ]:

prepared_path = Path("../../data/df_dynamics_with_oof.csv")
df_prepared = pd.read_csv(prepared_path)

K_SUN, K_WIND = 8.5, 14.0               # coeficients T_env calibrats
SUN_MAP={'In full sun':1.0,'In a mixture of sun and shadow':0.5,'In full shade':0.0}
WIND_MAP={"It's not windy":0.0,"A very light wind":0.25,"A light wind":0.5,
          "A moderate wind":0.75,"A strong wind":1.0}

df1 = df_prepared.copy()
df1=df1.dropna(subset=['ID','stop_idx','walk_id','comfort3','<T-T_fixed+<T>>','sun','wind']).copy()
df1['sun_s']=df1['sun'].map(SUN_MAP); df1['wind_s']=df1['wind'].map(WIND_MAP)
df1['T_corr']=df1['<T-T_fixed+<T>>']
df1['T_env']=df1['T_corr']+K_SUN*df1['sun_s']-K_WIND*df1['wind_s']
df1=df1.dropna(subset=['sun_s','wind_s']).reset_index(drop=True)
print(f"vots={len(df1)}  participants={df1['ID'].nunique()}  caminades={df1['walk_id'].nunique()}")
print("estats:",df1['comfort3'].value_counts().to_dict())


print(df1.shape)
df1.columns


In [ ]:
state_order = ["comfortable", "neutral", "uncomfortable"]
state_num_map = {s: i for i, s in enumerate(state_order)}




def build_transitions_for_trap_sweep(
    df_prepared,
    state_col = "comfort3"  ,
    t_col = "T_env",
    hdx_col = "<HDX-HDX_fixed+<HDX>>",
    max_stop=5,
):
    df = df_prepared.copy()

    sort_cols = [c for c in ["ID", "walk_id", "stop_idx", "row_id"] if c in df.columns]
    group_cols = [c for c in ["ID", "walk_id"] if c in df.columns]

    df = df.sort_values(sort_cols).reset_index(drop=True)
    g = df.groupby(group_cols, sort=False)

    df["state"] = df[state_col]
    df["state_num"] = df["state"].map(state_num_map)

    df["next_state"] = g["state"].shift(-1)
    df["next_state_num"] = g["state_num"].shift(-1)
    df["next_stop_idx"] = g["stop_idx"].shift(-1)

    df["step_gap"] = df["next_stop_idx"] - df["stop_idx"]

    # T variables
    if t_col in df.columns:
        df["T_t"] = df[t_col]
        df["T_t1"] = g[t_col].shift(-1)
        df["T_mean"] = 0.5 * (df["T_t"] + df["T_t1"])
        df["T_delta"] = df["T_t1"] - df["T_t"]

    # HDX variables
    if hdx_col in df.columns:
        df["HDX_t"] = df[hdx_col]
        df["HDX_t1"] = g[hdx_col].shift(-1)
        df["HDX_mean"] = 0.5 * (df["HDX_t"] + df["HDX_t1"])
        df["HDX_delta"] = df["HDX_t1"] - df["HDX_t"]

    transitions = df[
        (df["step_gap"] == 1) &
        (df["stop_idx"] <= max_stop) &
        (df["next_stop_idx"] <= max_stop)
    ].copy()

    transitions = transitions.dropna(subset=["state", "next_state"]).copy()
    transitions["delta_state"] = transitions["next_state_num"] - transitions["state_num"]

    return transitions

In [ ]:
transitions = build_transitions_for_trap_sweep(
    df1,
    max_stop=5
)

transitions.shape

In [ ]:
# canviar-ho tot en el format de la Sensació


def make_event_df(transitions, metric):
    df = transitions.copy()

    if metric == "neutralization_of_uncomfortable":
        # P( U-> N)
        sub = df[df["state"] == "uncomfortable"].copy()
        sub["event"] = (sub["next_state"] == "neutral").astype(int)

    # elif metric == "u_persistence":
    #     # P(U -> U)
    #     sub = df[df["state"] == "uncomfortable"].copy()
    #     sub["event"] = (sub["next_state"] == "uncomfortable").astype(int)

    # elif metric == "u_adverse_persistence":
    #     # P(U -> U or VU)
    #     sub = df[df["state"] == "uncomfortable"].copy()
    #     sub["event"] = sub["next_state"].isin([
    #         "uncomfortable",
    #         "very uncomfortable"
    #     ]).astype(int)

    elif metric == "break_from_neutral":
        # P(next in {U,C} | current in {N})
        sub = df[df["state"].isin([
            "neutral",
        ])].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
            "comfortable"
        ]).astype(int)

    elif metric == "recovery_to_neutral":
        # P(next = N | current in {U})
        sub = df[df["state"].isin([
            "uncomfortable",
        ])].copy()
        sub["event"] = (sub["next_state"] == "neutral").astype(int)

    elif metric == "recovery_to_comfortable":
        # P(next = C | current in {U,VU})
        sub = df[df["state"].isin([
            "uncomfortable",
        ])].copy()
        sub["event"] = (sub["next_state"] == "comfortable").astype(int)

    elif metric == "onset_from_comfortable":
        # P(next in {U,VU} | current = CN)
        sub = df[df["state"] == "comfortable"].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
        ]).astype(int)
        
    elif metric == "onset_from_neutral":
        # P(next in {U,VU} | current = CN)
        sub = df[df["state"] == "neutral"].copy()
        sub["event"] = sub["next_state"].isin([
            "uncomfortable",
        ]).astype(int)

    elif metric == "onset_of_neutrality":
        # P(next in {N} | current = C)
        sub = df[df["state"] == "comfortable"].copy()
        sub["event"] = (sub["next_state"] == "neutral").astype(int)

    else:
        raise ValueError(f"Unknown metric: {metric}")

    return sub

In [ ]:
metrics_to_test = [
    "neutralization_of_uncomfortable",
    "break_from_neutral",
    "recovery_to_neutral",
    "recovery_to_comfortable",
    "onset_from_comfortable",
    "onset_from_neutral",
    "onset_of_neutrality"
]

x_cols_to_test = [
    # "HDX_mean",
    # "HDX_t1",
    # "HDX_delta",
    # "HDX_t",
    "T_t",
    "T_delta",
    "T_mean",
    "T_t1",
]

In [ ]:
def bootstrap_threshold_sweep(
    transitions,
    x_col,
    metric,
    direction="above",
    n_grid=60,
    min_at_risk=20,
    q_low=0.05,
    q_high=0.95,
    n_boot=1000,
    seed=123,
    cluster_cols=("ID", "walk_id"),
):
    rng = np.random.default_rng(seed)

    event_df = make_event_df(transitions, metric)
    event_df = event_df.dropna(subset=[x_col, "event"]).copy()

    if event_df.empty:
        return pd.DataFrame()

    # trajectory / participant-walk cluster id
    cluster_cols = [c for c in cluster_cols if c in event_df.columns]

    if len(cluster_cols) == 0:
        event_df["_cluster_id"] = event_df.index.astype(str)
    else:
        event_df["_cluster_id"] = (
            event_df[cluster_cols].astype(str).agg("||".join, axis=1)
        )

    # Use fixed thresholds from observed data
    thresholds = np.quantile(
        event_df[x_col].to_numpy(),
        np.linspace(q_low, q_high, n_grid)
    )
    thresholds = np.unique(thresholds)

    # Observed curve
    observed_rows = []

    for thr in thresholds:
        if direction == "above":
            sub = event_df[event_df[x_col] >= thr]
        elif direction == "below":
            sub = event_df[event_df[x_col] <= thr]
        else:
            raise ValueError("direction must be 'above' or 'below'")

        n = len(sub)

        if n < min_at_risk:
            continue

        k = int(sub["event"].sum())
        observed_rows.append({
            "threshold": thr,
            "n_at_risk": n,
            "events": k,
            "prob_observed": k / n,
        })

    observed = pd.DataFrame(observed_rows)

    if observed.empty:
        return pd.DataFrame()

    valid_thresholds = observed["threshold"].to_numpy()

    # Pre-split by cluster for faster bootstrap
    cluster_ids = event_df["_cluster_id"].drop_duplicates().to_numpy()
    cluster_groups = {
        cid: g.drop(columns=["_cluster_id"])
        for cid, g in event_df.groupby("_cluster_id", sort=False)
    }

    boot_rows = []

    for b in range(n_boot):
        sampled_ids = rng.choice(cluster_ids, size=len(cluster_ids), replace=True)

        boot_df = pd.concat(
            [cluster_groups[cid] for cid in sampled_ids],
            ignore_index=True
        )

        for thr in valid_thresholds:
            if direction == "above":
                sub = boot_df[boot_df[x_col] >= thr]
            else:
                sub = boot_df[boot_df[x_col] <= thr]

            n = len(sub)

            if n < min_at_risk:
                boot_rows.append({
                    "boot": b,
                    "threshold": thr,
                    "prob_boot": np.nan,
                    "n_boot_at_risk": n,
                })
                continue

            k = int(sub["event"].sum())
            boot_rows.append({
                "boot": b,
                "threshold": thr,
                "prob_boot": k / n,
                "n_boot_at_risk": n,
            })

    boot = pd.DataFrame(boot_rows)

    summary = (
        boot
        .dropna(subset=["prob_boot"])
        .groupby("threshold")
        .agg(
            q025=("prob_boot", lambda x: np.quantile(x, 0.025)),
            median=("prob_boot", lambda x: np.quantile(x, 0.50)),
            q975=("prob_boot", lambda x: np.quantile(x, 0.975)),
            boot_valid=("prob_boot", "count"),
            boot_n_median=("n_boot_at_risk", "median"),
        )
        .reset_index()
    )

    out = observed.merge(summary, on="threshold", how="left")

    out["metric"] = metric
    out["x_col"] = x_col
    out["direction"] = direction
    out["condition"] = out.apply(
        lambda r: f"{x_col} >= {r['threshold']:.3f}"
        if direction == "above"
        else f"{x_col} <= {r['threshold']:.3f}",
        axis=1
    )

    # Useful ranking criterion
    out["score_boot_q025"] = out["q025"]

    out = out.sort_values(
        ["score_boot_q025", "prob_observed", "n_at_risk"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    return out

In [ ]:
bootstrap_threshold_results = []

for metric in metrics_to_test:
    for x_col in x_cols_to_test:
        if x_col in transitions.columns:
            tmp = bootstrap_threshold_sweep(
                transitions=transitions,
                x_col=x_col,
                metric=metric,
                direction="below",
                n_grid=60,
                min_at_risk=20,
                q_low=0.05,
                q_high=0.95,
                n_boot=1000,
                seed=123,
                cluster_cols=("ID", "walk_id"),
            )
            bootstrap_threshold_results.append(tmp)

bootstrap_threshold_results = pd.concat(
    bootstrap_threshold_results,
    ignore_index=True
)

bootstrap_threshold_results.to_csv(
    "markov_newT/binary_bootstrap_threshold_sweep_T_HDX_below.csv",
    index=False
)



# ============================================================
# ABOVE
# ============================================================

bootstrap_threshold_results_above_list = []

for metric in metrics_to_test:
    for x_col in x_cols_to_test:
        if x_col in transitions.columns:
            print(f"Running ABOVE | metric={metric} | x_col={x_col}")

            tmp = bootstrap_threshold_sweep(
                transitions=transitions,
                x_col=x_col,
                metric=metric,
                direction="above",
                n_grid=60,
                min_at_risk=20,
                q_low=0.05,
                q_high=0.95,
                n_boot=1000,
                seed=123,
                cluster_cols=("ID", "walk_id"),
            )

            if not tmp.empty:
                bootstrap_threshold_results_above_list.append(tmp)

bootstrap_threshold_results_above = pd.concat(
    bootstrap_threshold_results_above_list,
    ignore_index=True
)

bootstrap_threshold_results_above.to_csv(
    "markov_newT/binary_bootstrap_threshold_sweep_T_HDX_above.csv",
    index=False
)

bootstrap_threshold_results.head(20)

bootstrap_threshold_results_above = pd.read_csv("markov_newT/binary_bootstrap_threshold_sweep_T_HDX_above.csv")
bootstrap_threshold_results_below = pd.read_csv("markov_newT/binary_bootstrap_threshold_sweep_T_HDX_below.csv") 


# si la fem per HDX bins

"T_corr_start"   # temperatura de sortida
"T_corr_end"     # temperatura d'arribada
"T_corr_mean"    # temperatura mitjana de la transició
"T_corr_delta"   # canvi de temperatura

"T_env_start"
"T_env_end"
"T_env_mean"
"T_env_delta"

In [ ]:

transitions = transitions.copy()



    

transitions["T_bin"] = pd.cut(
    transitions["T_corr_end"],
    bins=[-float("inf"), 24, 30.8, float("inf")],
    labels=["lower_T", "neutral_T", "higher_T"],
    right=False
)

# transitions["T_bin"] = pd.cut(
#     transitions["T_corr"],
#     bins=[-float("inf"), 26.1, 30.7, float("inf")],
#     labels=["lower_T", "neutral_T", "higher_T"],
#     right=False
# )


# transitions["T_bin"] = pd.cut(
#     transitions["T_next2"],
#     bins=[-float("inf"), 24.62, 30.8, float("inf")],
#     labels=["lower_T", "neutral_T", "higher_T"],
#     right=False
# )


def plot_transition_matrix(cm_prob, title="Matriu de transició"):
    labels = list(cm_prob.columns)

    fig, ax = plt.subplots(figsize=(7.5, 5.8), constrained_layout=True)
    im = ax.imshow(cm_prob.values, aspect="auto", cmap="Reds", vmin=0, vmax=1)

    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=28, ha="right")
    ax.set_yticklabels(labels)

    ax.set_xlabel("Estat següent")
    ax.set_ylabel("Estat actual")
    ax.set_title(title)

    for i in range(cm_prob.shape[0]):
        for j in range(cm_prob.shape[1]):
            ax.text(
                j, i,
                f"{cm_prob.iloc[i, j]:.3f}",
                ha="center", va="center", fontsize=10
            )

    cbar = fig.colorbar(im, ax=ax, shrink=0.95)
    cbar.set_label("Probabilitat")

    plt.show()

tmats_T = {}

for b, sub in transitions.groupby("T_bin"):
    cm = pd.crosstab(
        sub["state"],
        sub["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    cm_prob = cm.div(cm.sum(axis=1), axis=0)
    tmats_T[b] = cm_prob

    cm_prob.to_csv(f"csvs_proces/cm_prob_{b}.csv")
    print(f"\n{b}")
    print(cm_prob)
    plot_transition_matrix(cm_prob, title=f"Matriu de transició - {b}")

print(transitions.groupby("T_bin").size())
print(transitions.groupby(["T_bin", "state"])["next_state"].count())

In [ ]:
import numpy as np
import pandas as pd

# Exemple amb thresholds. Canvia'ls pels teus règims reals.
bins = [-np.inf, 27, 30.5, np.inf]
labels = ["cool", "central", "hot"]

transitions["T_start_bin"] = pd.cut(
    transitions["T_env_start"],
    bins=bins,
    labels=labels,
    right=False
)

transitions["T_end_bin"] = pd.cut(
    transitions["T_env_end"],
    bins=bins,
    labels=labels,
    right=False
)

# Counts absoluts
counts = pd.crosstab(
    transitions["T_start_bin"],
    transitions["T_end_bin"]
)

# Probabilitat d'arribada donada la sortida
row_norm = pd.crosstab(
    transitions["T_start_bin"],
    transitions["T_end_bin"],
    normalize="index"
)

counts, row_norm

In [ ]:
onset_risk = transitions[transitions["state"] == "comfortable"].copy()

pd.crosstab(
    onset_risk["T_start_bin"],
    onset_risk["T_end_bin"],
    normalize="index"
)

In [ ]:
import numpy as np
import pandas as pd

state_order = ["comfortable", "neutral", "uncomfortable"]

def add_start_end_bins(
    transitions,
    start_col,
    end_col,
    bins,
    labels=("cool", "central", "hot"),
    prefix="T"
):
    """
    Adds start/end thermal regime bins to transitions.
    """
    transitions = transitions.copy()

    transitions[f"{prefix}_start_bin"] = pd.cut(
        transitions[start_col],
        bins=bins,
        labels=labels,
        right=False
    )

    transitions[f"{prefix}_end_bin"] = pd.cut(
        transitions[end_col],
        bins=bins,
        labels=labels,
        right=False
    )

    return transitions


def transition_matrix(sub, state_order=state_order):
    """
    Row-normalised transition matrix P(s_{i+1} | s_i).
    """
    counts = pd.crosstab(
        sub["state"],
        sub["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    probs = counts.div(counts.sum(axis=1), axis=0)

    return counts, probs


def analyse_same_regime_transitions(
    transitions,
    start_bin_col,
    end_bin_col,
    regimes=("cool", "central", "hot"),
    state_order=state_order
):
    """
    For each regime r, keeps only transitions with:
        T_i bin = r and T_{i+1} bin = r

    Returns counts/probability matrices and sample sizes.
    """
    results = {}

    summary_rows = []

    for r in regimes:
        sub = transitions[
            (transitions[start_bin_col] == r) &
            (transitions[end_bin_col] == r)
        ].copy()

        counts, probs = transition_matrix(sub, state_order=state_order)

        results[r] = {
            "data": sub,
            "counts": counts,
            "probs": probs
        }

        summary_rows.append({
            "regime": r,
            "n_transitions": len(sub),
            "n_walks": sub[["ID", "walk_id"]].drop_duplicates().shape[0],
            "n_participants": sub["ID"].nunique(),
            "C_to_C": probs.loc["comfortable", "comfortable"],
            "C_to_U": probs.loc["comfortable", "uncomfortable"],
            "U_to_C": probs.loc["uncomfortable", "comfortable"],
            "U_to_U": probs.loc["uncomfortable", "uncomfortable"],
        })

    summary = pd.DataFrame(summary_rows)

    return results, summary

In [ ]:
Tcorr_bins = [-np.inf, 26.10, 30.77, np.inf]
labels = ["cool", "central", "hot"]

transitions_corr = add_start_end_bins(
    transitions,
    start_col="T_corr_start",
    end_col="T_corr_end",
    bins=Tcorr_bins,
    labels=labels,
    prefix="Tcorr"
)

results_corr_same, summary_corr_same = analyse_same_regime_transitions(
    transitions_corr,
    start_bin_col="Tcorr_start_bin",
    end_bin_col="Tcorr_end_bin",
    regimes=labels
)

summary_corr_same

In [ ]:
Tenv_bins = [-np.inf, 24.62, 30.83, np.inf]

transitions_env = add_start_end_bins(
    transitions,
    start_col="T_env_start",
    end_col="T_env_end",
    bins=Tenv_bins,
    labels=labels,
    prefix="Tenv"
)

results_env_same, summary_env_same = analyse_same_regime_transitions(
    transitions_env,
    start_bin_col="Tenv_start_bin",
    end_bin_col="Tenv_end_bin",
    regimes=labels
)

summary_env_same

In [ ]:
import numpy as np
import pandas as pd

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))

    lo = (centre - half) / denom
    hi = (centre + half) / denom

    return lo, hi


def transition_channels_with_wilson(
    sub,
    state_col="state",
    next_col="next_state",
    state_order=("comfortable", "neutral", "uncomfortable")
):
    rows = []

    for s0 in state_order:
        row = sub[sub[state_col] == s0]
        n = len(row)

        for s1 in state_order:
            k = (row[next_col] == s1).sum()
            p = k / n if n > 0 else np.nan
            lo, hi = wilson_ci(k, n)

            rows.append({
                "from": s0,
                "to": s1,
                "k": k,
                "n": n,
                "p": p,
                "ci_low": lo,
                "ci_high": hi
            })

    return pd.DataFrame(rows)

In [ ]:
all_ci_rows = []

for coord, results in [
    ("Tcorr", results_corr_same),
    ("Tenv", results_env_same)
]:
    for regime, obj in results.items():
        ci = transition_channels_with_wilson(obj["data"])
        ci["coordinate"] = coord
        ci["regime"] = regime
        all_ci_rows.append(ci)

same_regime_ci = pd.concat(all_ci_rows, ignore_index=True)

same_regime_ci[
    (same_regime_ci["from"].isin(["comfortable", "uncomfortable"])) &
    (same_regime_ci["to"].isin(["comfortable", "uncomfortable"]))
][
    ["coordinate", "regime", "from", "to", "k", "n", "p", "ci_low", "ci_high"]
]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

In [ ]:
def paper_style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "mathtext.fontset": "dejavusans",
        "font.size": 12,
        "axes.labelsize": 14,
        "axes.titlesize": 16,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
    })

paper_style()

blue_cmap = LinearSegmentedColormap.from_list(
    "comfort_blues",
    ["#f7fbff", "#deebf7", "#9ecae1", "#3182bd", "#08519c", "#08306b"]
)

In [ ]:
state_order = ["comfortable", "neutral", "uncomfortable"]
state_labels = ["C", "N", "U"]

regime_labels = ["lower_T", "neutral_T", "higher_T"]
regime_titles = {
    "lower_T": "Cool",
    "neutral_T": "Central",
    "higher_T": "Hot"
}


def compute_transition_matrix(sub, state_order=state_order):
    cm = pd.crosstab(
        sub["state"],
        sub["next_state"]
    ).reindex(index=state_order, columns=state_order, fill_value=0)

    cm_prob = cm.div(cm.sum(axis=1), axis=0)

    return cm, cm_prob

In [ ]:
def add_arrival_bins(transitions):
    transitions = transitions.copy()

    transitions["Tcorr_start_bin"] = pd.cut(
        transitions["T_corr_start"],
        bins=[-np.inf, 26.10, 30.77, np.inf],
        labels=regime_labels,
        right=False
    )

    transitions["Tenv_start_bin"] = pd.cut(
        transitions["T_env_start"],
        bins=[-np.inf, 24.62, 30.83, np.inf],
        labels=regime_labels,
        right=False
    )

    return transitions

In [ ]:
def plot_arrival_transition_matrices_blue(
    transitions,
    output_pdf="transition_matrices_start_blue.pdf",
    output_png="transition_matrices_start_blue.png",
    vmin=0,
    vmax=0.85
):
    transitions = add_arrival_bins(transitions)

    coord_info = [
        {
            "name": "Tcorr",
            "label": r"$T_{\mathrm{corr},i+1}$",
            "bin_col": "Tcorr_start_bin"
        },
        {
            "name": "Tenv",
            "label": r"$T_{\mathrm{env},i+1}$",
            "bin_col": "Tenv_start_bin"
        },
    ]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(7.2, 4.6),
        sharex=True,
        sharey=True
    )

    norm = Normalize(vmin=vmin, vmax=vmax)

    for row, coord in enumerate(coord_info):
        for col, regime in enumerate(regime_labels):
            ax = axes[row, col]

            sub = transitions[transitions[coord["bin_col"]] == regime].copy()
            cm, cm_prob = compute_transition_matrix(sub)

            mat = cm_prob.values.astype(float)

            im = ax.imshow(
                mat,
                aspect="equal",
                cmap=blue_cmap,
                norm=norm
            )

            ax.set_xticks(range(len(state_labels)))
            ax.set_yticks(range(len(state_labels)))
            ax.set_xticklabels(state_labels)
            ax.set_yticklabels(state_labels)

            if row == 0:
                ax.set_title(regime_titles[regime], pad=8)

            if col == 0:
                ax.set_ylabel(coord["label"] + "\n" + r"$s_i$")

            if row == 1:
                ax.set_xlabel(r"$s_{i+1}$")

            ax.tick_params(axis="both", length=0)

            # Grid blanc entre cel·les
            ax.set_xticks(np.arange(-0.5, len(state_labels), 1), minor=True)
            ax.set_yticks(np.arange(-0.5, len(state_labels), 1), minor=True)
            ax.grid(which="minor", color="white", linewidth=1.0)
            ax.tick_params(which="minor", bottom=False, left=False)

            # N de transicions
            ax.text(
                0.03,
                0.97,
                rf"$n={len(sub)}$",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
                bbox=dict(
                    boxstyle="round,pad=0.15",
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.85
                )
            )

            # Valors dins la matriu
            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    p = mat[i, j]
                    k = cm.values[i, j]

                    if np.isnan(p):
                        txt = ""
                    else:
                        txt = f"{p:.2f}\n({k})"

                    text_color = "white" if p >= 0.50 else "black"

                    ax.text(
                        j,
                        i,
                        txt,
                        ha="center",
                        va="center",
                        fontsize=8.5,
                        color=text_color
                    )

    # Colorbar comuna
    cbar_ax = fig.add_axes([0.91, 0.18, 0.018, 0.66])
    sm = ScalarMappable(norm=norm, cmap=blue_cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label(r"$P(s_{i+1}\mid s_i)$", rotation=90, labelpad=10)

    fig.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.11,
        top=0.90,
        wspace=0.18,
        hspace=0.25
    )

    fig.savefig(output_pdf)
    fig.savefig(output_png, dpi=300)
    plt.show()

    return fig

In [ ]:
def plot_arrival_transition_matrices_blue(
    transitions,
    output_pdf="transition_matrices_start_blue.pdf",
    output_png="transition_matrices_start_blue.png",
    vmin=0,
    vmax=0.85
):
    transitions = add_arrival_bins(transitions)

    coord_info = [
        {
            "name": "Tcorr",
            "label": r"$T_{\mathrm{corr}}$ (i+1)",
            "bin_col": "Tcorr_start_bin"
        },
        {
            "name": "Tenv",
            "label": r"$T_{\mathrm{env}}$ (i+1)",
            "bin_col": "Tenv_start_bin"
        },
    ]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(7.2, 4.6),
        sharex=True,
        sharey=True
    )

    norm = Normalize(vmin=vmin, vmax=vmax)

    for row, coord in enumerate(coord_info):
        for col, regime in enumerate(regime_labels):
            ax = axes[row, col]

            sub = transitions[transitions[coord["bin_col"]] == regime].copy()
            cm, cm_prob = compute_transition_matrix(sub)

            mat = cm_prob.values.astype(float)

            im = ax.imshow(
                mat,
                aspect="equal",
                cmap=blue_cmap,
                norm=norm
            )

            ax.set_xticks(range(len(state_labels)))
            ax.set_yticks(range(len(state_labels)))
            ax.set_xticklabels(state_labels)
            ax.set_yticklabels(state_labels)

            if row == 0:
                ax.set_title(regime_titles[regime], pad=8)

            if col == 0:
                ax.set_ylabel(coord["label"] + "\n" + r"$s_i$")

            if row == 1:
                ax.set_xlabel(r"$s_{i+1}$")

            ax.tick_params(axis="both", length=0)

            # Grid blanc entre cel·les
            # ax.set_xticks(np.arange(-0.5, len(state_labels), 1), minor=True)
            # ax.set_yticks(np.arange(-0.5, len(state_labels), 1), minor=True)
            # ax.grid(which="minor", color="white", linewidth=1.0)
            # ax.tick_params(which="minor", bottom=False, left=False)

            

            # Valors dins la matriu
            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    p = mat[i, j]
                    k = cm.values[i, j]

                    if np.isnan(p):
                        txt = ""
                    else:
                        txt = f"{p:.2f}\n({k})"

                    text_color = "white" if p >= 0.50 else "black"

                    ax.text(
                        j,
                        i,
                        txt,
                        ha="center",
                        va="center",
                        fontsize=8.5,
                        color=text_color
                    )

    # Colorbar comuna
    cbar_ax = fig.add_axes([0.91, 0.18, 0.018, 0.66])
    sm = ScalarMappable(norm=norm, cmap=blue_cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label(r"$P(s_{i+1}\mid s_i)$", rotation=90, labelpad=10)

    fig.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.11,
        top=0.90,
        wspace=0.18,
        hspace=0.25
    )

    fig.savefig(output_pdf)
    fig.savefig(output_png, dpi=300)
    plt.show()

    return fig

In [ ]:
fig = plot_arrival_transition_matrices_blue(transitions)

In [ ]:
def paper_style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "mathtext.fontset": "dejavusans",
        "font.size": 13,
        "axes.labelsize": 15,
        "axes.titlesize": 17,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
    })

paper_style()

In [ ]:
def add_start_end_bins(transitions):
    transitions = transitions.copy()

    transitions["Tcorr_start_bin"] = pd.cut(
        transitions["T_corr_start"],
        bins=[-np.inf, 26.10, 30.77, np.inf],
        labels=regime_labels,
        right=False
    )

    transitions["Tcorr_end_bin"] = pd.cut(
        transitions["T_corr_end"],
        bins=[-np.inf, 26.10, 30.77, np.inf],
        labels=regime_labels,
        right=False
    )

    transitions["Tenv_start_bin"] = pd.cut(
        transitions["T_env_start"],
        bins=[-np.inf, 24.62, 30.83, np.inf],
        labels=regime_labels,
        right=False
    )

    transitions["Tenv_end_bin"] = pd.cut(
        transitions["T_env_end"],
        bins=[-np.inf, 24.62, 30.83, np.inf],
        labels=regime_labels,
        right=False
    )

    return transitions

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))

    lo = (centre - half) / denom
    hi = (centre + half) / denom

    return lo, hi


def channel_prob_with_ci(sub, from_state, to_state):
    risk = sub[sub["state"] == from_state]
    n = len(risk)

    if n == 0:
        return np.nan, np.nan, np.nan, 0, 0

    k = (risk["next_state"] == to_state).sum()
    p = k / n
    lo, hi = wilson_ci(k, n)

    return p, lo, hi, k, n

In [ ]:
def compute_same_regime_channels(transitions):
    transitions = add_start_end_bins(transitions)

    coords = {
        "Tcorr": {
            "label": r"$T_{\mathrm{corr}}$",
            "start_bin": "Tcorr_start_bin",
            "end_bin": "Tcorr_end_bin"
        },
        "Tenv": {
            "label": r"$T_{\mathrm{env}}$",
            "start_bin": "Tenv_start_bin",
            "end_bin": "Tenv_end_bin"
        }
    }

    channels = [
        ("comfortable", "comfortable", "C_to_C", r"$C\to C$"),
        ("uncomfortable", "uncomfortable", "U_to_U", r"$U\to U$"),
        ("comfortable", "uncomfortable", "C_to_U", r"$C\to U$"),
        ("uncomfortable", "comfortable", "U_to_C", r"$U\to C$")
    ]

    rows = []

    for coord, cfg in coords.items():
        for regime in regime_labels:
            sub = transitions[
                (transitions[cfg["start_bin"]] == regime) &
                (transitions[cfg["end_bin"]] == regime)
            ].copy()

            for from_state, to_state, channel, channel_label in channels:
                p, lo, hi, k, n = channel_prob_with_ci(
                    sub,
                    from_state,
                    to_state
                )

                rows.append({
                    "coordinate": coord,
                    "coordinate_label": cfg["label"],
                    "regime": regime,
                    "regime_title": regime_titles[regime],
                    "channel": channel,
                    "channel_label": channel_label,
                    "from": from_state,
                    "to": to_state,
                    "k": k,
                    "n": n,
                    "p": p,
                    "ci_low": lo,
                    "ci_high": hi,
                    "n_transitions": len(sub)
                })

    return pd.DataFrame(rows)

In [ ]:
same_regime_df = compute_same_regime_channels(transitions)

same_regime_df[
    same_regime_df["channel"].isin(["C_to_C", "U_to_U"])
][
    ["coordinate", "regime", "channel", "k", "n", "p", "ci_low", "ci_high", "n_transitions"]
]

In [ ]:
def plot_same_regime_persistence_blue(
    same_regime_df,
    output_pdf="same_regime_persistence_blue.pdf",
    output_png="same_regime_persistence_blue.png"
):
    plot_df = same_regime_df[
        same_regime_df["channel"].isin(["C_to_C", "U_to_U"])
    ].copy()

    coords = ["Tcorr", "Tenv"]
    coord_titles = {
        "Tcorr": r"$T_{\mathrm{corr}}$",
        "Tenv": r"$T_{\mathrm{env}}$"
    }

    x = np.arange(len(regime_labels))

    offsets = {
        "C_to_C": -0.07,
        "U_to_U": +0.07
    }

    colors = {
        "C_to_C": "#6BAED6",
        "U_to_U": "#08519C"
    }

    markers = {
        "C_to_C": "o",
        "U_to_U": "s"
    }

    labels = {
        "C_to_C": r"$C\to C$",
        "U_to_U": r"$U\to U$"
    }

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(7.0, 3.0),
        sharey=True
    )

    for ax, coord in zip(axes, coords):
        sub_coord = plot_df[plot_df["coordinate"] == coord].copy()

        for channel in ["C_to_C", "U_to_U"]:
            sub_ch = (
                sub_coord[sub_coord["channel"] == channel]
                .set_index("regime")
                .reindex(regime_labels)
                .reset_index()
            )

            y = sub_ch["p"].values
            yerr_low = y - sub_ch["ci_low"].values
            yerr_high = sub_ch["ci_high"].values - y
            yerr = np.vstack([yerr_low, yerr_high])

            xpos = x + offsets[channel]

            ax.errorbar(
                xpos,
                y,
                yerr=yerr,
                fmt=markers[channel],
                color=colors[channel],
                markerfacecolor=colors[channel],
                markeredgecolor="black",
                markeredgewidth=0.4,
                elinewidth=1.0,
                capsize=3,
                markersize=5,
                linewidth=1.0,
                label=labels[channel],
                zorder=3
            )

            ax.plot(
                xpos,
                y,
                color=colors[channel],
                linewidth=1.0,
                alpha=0.95,
                zorder=2
            )

        # ax.axhline(
        #     0.5,
        #     color="0.75",
        #     linestyle="--",
        #     linewidth=0.8,
        #     zorder=1
        # )

        ax.set_title(coord_titles[coord])
        ax.set_xticks(x)
        ax.set_xticklabels([regime_titles[r] for r in regime_labels])
        ax.set_ylim(0, 0.9)
        ax.set_xlabel("Thermal regime")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        

        # n labels
        for xi, regime in zip(x, regime_labels):
            n_regime = sub_coord[
                sub_coord["regime"] == regime
            ]["n_transitions"].iloc[0]

            # ax.text(
            #     xi,
            #     -0.13,
            #     rf"$n={int(n_regime)}$",
            #     ha="center",
            #     va="top",
            #     fontsize=8,
            #     transform=ax.get_xaxis_transform()
            # )

    axes[0].set_ylabel("Persistence probability")

    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.03)
    )

    fig.subplots_adjust(
        left=0.09,
        right=0.98,
        bottom=0.23,
        top=0.82,
        wspace=0.14
    )

    fig.savefig(output_pdf)
    fig.savefig(output_png, dpi=300)
    plt.show()

    return fig

In [ ]:
fig = plot_same_regime_persistence_blue(same_regime_df)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan

    p = k / n
    denom = 1 + z**2 / n
    centre = p + z**2 / (2 * n)
    half = z * np.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))

    lo = (centre - half) / denom
    hi = (centre + half) / denom

    return lo, hi


def binned_persistence_by_arrival_temperature(
    transitions,
    temp_col,
    n_bins=6,
    min_risk=25,
):
    """
    Computes C->C and U->U as a function of arrival temperature.

    Parameters
    ----------
    transitions : pd.DataFrame
        Must contain state, next_state and temp_col.
    temp_col : str
        Arrival temperature column, e.g. "T_corr_end" or "T_env_end".
    n_bins : int
        Number of quantile bins.
    min_risk : int
        Minimum number of at-risk transitions per state per bin.

    Returns
    -------
    pd.DataFrame
    """

    df = transitions.dropna(subset=[temp_col, "state", "next_state"]).copy()

    df["T_bin"] = pd.qcut(
        df[temp_col],
        q=n_bins,
        duplicates="drop"
    )

    rows = []

    channels = [
        ("comfortable", "comfortable", "C_to_C"),
        ("uncomfortable", "uncomfortable", "U_to_U"),
    ]

    for tbin, sub in df.groupby("T_bin", observed=True):
        t_mid = sub[temp_col].median()
        t_min = sub[temp_col].min()
        t_max = sub[temp_col].max()

        for from_state, to_state, channel in channels:
            risk = sub[sub["state"] == from_state]
            n = len(risk)
            k = (risk["next_state"] == to_state).sum()

            if n == 0:
                p = np.nan
                lo, hi = np.nan, np.nan
            else:
                p = k / n
                lo, hi = wilson_ci(k, n)

            rows.append({
                "temp_col": temp_col,
                "T_bin": str(tbin),
                "T_min": t_min,
                "T_max": t_max,
                "T_mid": t_mid,
                "channel": channel,
                "from": from_state,
                "to": to_state,
                "k": k,
                "n_risk": n,
                "p": p,
                "ci_low": lo,
                "ci_high": hi,
                "valid": n >= min_risk,
            })

    return pd.DataFrame(rows)

In [ ]:
pers_corr = binned_persistence_by_arrival_temperature(
    transitions,
    temp_col="T_corr_end",
    n_bins=6,
    min_risk=25,
)

pers_corr["coordinate"] = "Tcorr"

pers_env = binned_persistence_by_arrival_temperature(
    transitions,
    temp_col="T_env_end",
    n_bins=6,
    min_risk=25,
)

pers_env["coordinate"] = "Tenv"

pers_bins = pd.concat([pers_corr, pers_env], ignore_index=True)

pers_bins[
    ["coordinate", "T_mid", "channel", "k", "n_risk", "p", "ci_low", "ci_high", "valid"]
]

In [ ]:
def plot_binned_persistence(
    pers_bins,
    coordinate,
    title,
    output=None,
):
    sub = pers_bins[pers_bins["coordinate"] == coordinate].copy()

    colors = {
        "C_to_C": "#6BAED6",
        "U_to_U": "#08519C",
    }

    markers = {
        "C_to_C": "o",
        "U_to_U": "s",
    }

    labels = {
        "C_to_C": r"$C\to C$",
        "U_to_U": r"$U\to U$",
    }

    fig, ax = plt.subplots(figsize=(3.5, 2.8))

    for channel in ["C_to_C", "U_to_U"]:
        ch = sub[sub["channel"] == channel].sort_values("T_mid").copy()

        y = ch["p"].values
        x = ch["T_mid"].values

        yerr = np.vstack([
            y - ch["ci_low"].values,
            ch["ci_high"].values - y
        ])

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            fmt=markers[channel],
            color=colors[channel],
            markerfacecolor=colors[channel],
            markeredgecolor="black",
            markeredgewidth=0.4,
            linewidth=1.0,
            elinewidth=0.9,
            capsize=3,
            markersize=5,
            label=labels[channel],
            zorder=3,
        )

        ax.plot(
            x,
            y,
            color=colors[channel],
            linewidth=1.0,
            alpha=0.9,
            zorder=2,
        )

        # Mark bins with low risk
        low = ch[~ch["valid"]]
        if len(low) > 0:
            ax.scatter(
                low["T_mid"],
                low["p"],
                facecolors="none",
                edgecolors=colors[channel],
                s=70,
                linewidth=1.1,
                zorder=4,
            )

    ax.axhline(
        0.5,
        color="0.75",
        linestyle="--",
        linewidth=0.8,
        zorder=1,
    )

    ax.set_title(title)
    ax.set_xlabel("Arrival temperature")
    ax.set_ylabel("Persistence probability")
    ax.set_ylim(0, 0.9)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.7,
        zorder=0,
    )

    ax.legend(frameon=False, loc="best")

    fig.tight_layout()

    if output is not None:
        fig.savefig(output)
        fig.savefig(output.replace(".pdf", ".png"), dpi=300)

    plt.show()

    return fig

# mirem amb diferents colors

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np

# Paleta TFM
PAL_RED   = "#F52837"
PAL_MINT  = "#9CE5E8"
PAL_BLUE  = "#2C7AB5"
PAL_NAVY  = "#152F61"

# Colormap seqüencial per probabilitats
TFM_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "tfm_prob",
    ["#FFFFFF", PAL_MINT, PAL_BLUE, PAL_NAVY]
)

In [ ]:
plot_binned_persistence(
    pers_bins,
    coordinate="Tcorr",
    title=r"$T_{\mathrm{corr},i+1}$",
    output="binned_persistence_Tcorr_arrival.pdf"
)

plot_binned_persistence(
    pers_bins,
    coordinate="Tenv",
    title=r"$T_{\mathrm{env},i+1}$",
    output="binned_persistence_Tenv_arrival.pdf"
)

In [ ]:
def compute_route_geometry_matrix(
    transitions,
    start_bin_col,
    end_bin_col,
    regime_labels=regime_labels
):
    counts = pd.crosstab(
        transitions[start_bin_col],
        transitions[end_bin_col]
    ).reindex(
        index=regime_labels,
        columns=regime_labels,
        fill_value=0
    )

    probs = counts.div(counts.sum(axis=1), axis=0)

    return counts, probs

In [ ]:
def plot_route_geometry_blue(
    transitions,
    output_pdf="route_geometry_Ti_to_Tip1_blue.pdf",
    output_png="route_geometry_Ti_to_Tip1_blue.png",
    onset_risk_only=False,
    vmin=0,
    vmax=1
):
    transitions = add_start_end_bins(transitions)

    if onset_risk_only:
        transitions = transitions[transitions["state"] == "comfortable"].copy()
        title_suffix = "onset-risk"
    else:
        title_suffix = "all transitions"

    coord_info = [
        {
            "name": "Tcorr",
            "title": r"$T_{\mathrm{corr}}$",
            "start_bin": "Tcorr_start_bin",
            "end_bin": "Tcorr_end_bin"
        },
        {
            "name": "Tenv",
            "title": r"$T_{\mathrm{env}}$",
            "start_bin": "Tenv_start_bin",
            "end_bin": "Tenv_end_bin"
        }
    ]

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(6.8, 3.2),
        sharex=True,
        sharey=True
    )

    norm = Normalize(vmin=vmin, vmax=vmax)

    for ax, coord in zip(axes, coord_info):
        counts, probs = compute_route_geometry_matrix(
            transitions,
            start_bin_col=coord["start_bin"],
            end_bin_col=coord["end_bin"]
        )

        mat = probs.values.astype(float)

        im = ax.imshow(
            mat,
            aspect="equal",
            cmap=blue_cmap,
            norm=norm,
            interpolation="nearest"
        )

        ax.set_title(coord["title"], pad=8)

        ax.set_xticks(range(len(regime_labels)))
        ax.set_yticks(range(len(regime_labels)))

        ax.set_xticklabels(
            [regime_titles[r] for r in regime_labels],
            rotation=25,
            ha="right"
        )
        ax.set_yticklabels([regime_titles[r] for r in regime_labels])

        ax.set_xlabel(r"$T_{i+1}$ regime")

        ax.tick_params(axis="both", length=0)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                p = mat[i, j]
                k = counts.values[i, j]

                if np.isnan(p):
                    txt = ""
                else:
                    txt = f"{p:.2f}\n({k})"

                text_color = "white" if p >= 0.50 else "black"

                ax.text(
                    j,
                    i,
                    txt,
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    color=text_color
                )

        

    axes[0].set_ylabel(r"$T_i$ regime")

    cbar_ax = fig.add_axes([0.92, 0.20, 0.018, 0.62])
    sm = ScalarMappable(norm=norm, cmap=blue_cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label(
        r"$P(T_{i+1}\ \mathrm{bin}\mid T_i\ \mathrm{bin})$",
        rotation=90,
        labelpad=10
    )

    fig.subplots_adjust(
        left=0.10,
        right=0.89,
        bottom=0.20,
        top=0.84,
        wspace=0.18
    )

    fig.savefig(output_pdf)
    fig.savefig(output_png, dpi=300)
    plt.show()

    return fig

In [ ]:
fig_route = plot_route_geometry_blue(
    transitions,
    output_pdf="route_geometry_all_transitions_blue.pdf",
    output_png="route_geometry_all_transitions_blue.png",
    onset_risk_only=False
)

# i ara trobo R_D

In [ ]:
import numpy as np
import pandas as pd

transitions = transitions.copy()

state_order = ["comfortable", "neutral", "uncomfortable"]

bin_numbers = [12, 10, 8, 6, 5]


def safe_div(a, b):
    if b == 0:
        return np.nan
    return a / b


def compute_RD_from_transition_matrix(cm, cm_prob):
    """
    Computes two R_D definitions from a transition count matrix
    and its row-normalised probability matrix.

    Definitions with raw counts:
        R_D_strict_counts = #(C -> U) / #(U -> C)
        R_D_broad_counts  = #((C,N) -> U) / #(U -> (N,C))

    Definitions with conditional probabilities:
        R_D_strict_prob = P(U | C) / P(C | U)
        R_D_broad_prob  = P(U | C or N) / P(C or N | U)

    The count version is closer to the literal expression
    "number of transitions". The probability version controls for
    how many transitions were at risk in each starting state.
    """

    C = "comfortable"
    N = "neutral"
    U = "uncomfortable"

    # ========================================================
    # Raw counts
    # ========================================================

    k_CU = cm.loc[C, U]
    k_UC = cm.loc[U, C]

    k_CNU = cm.loc[C, U] + cm.loc[N, U]
    k_UCN = cm.loc[U, C] + cm.loc[U, N]

    RD_strict_counts = safe_div(k_CU, k_UC)
    RD_broad_counts = safe_div(k_CNU, k_UCN)

    # ========================================================
    # Row-normalised probabilities
    # ========================================================

    p_CU = cm_prob.loc[C, U]
    p_UC = cm_prob.loc[U, C]

    # For the broad version we need the probability of ending in U
    # given that the origin was C or N. This cannot be obtained by
    # simply averaging cm_prob rows; it must be weighted by counts.
    n_C = cm.loc[C].sum()
    n_N = cm.loc[N].sum()
    n_U = cm.loc[U].sum()

    p_CNU = safe_div(k_CNU, n_C + n_N)
    p_UCN = safe_div(k_UCN, n_U)

    RD_strict_prob = safe_div(p_CU, p_UC)
    RD_broad_prob = safe_div(p_CNU, p_UCN)

    return {
        "k_CU": k_CU,
        "k_UC": k_UC,
        "k_CNU": k_CNU,
        "k_UCN": k_UCN,

        "n_C_risk": n_C,
        "n_N_risk": n_N,
        "n_U_risk": n_U,
        "n_CN_risk": n_C + n_N,

        "p_CU": p_CU,
        "p_UC": p_UC,
        "p_CNU": p_CNU,
        "p_UCN": p_UCN,

        "RD_strict_counts": RD_strict_counts,
        "RD_broad_counts": RD_broad_counts,
        "RD_strict_prob": RD_strict_prob,
        "RD_broad_prob": RD_broad_prob,
    }

In [ ]:
all_RD_rows = []
all_tmats = {}

for n_bins in bin_numbers:

    transitions_tmp = transitions.copy()

    transitions_tmp["T_bin"] = pd.cut(
        transitions_tmp["T_corr_end"],
        bins=n_bins,
        include_lowest=True
    )

    tmats_T = {}

    print(f"\n\n==============================")
    print(f"{n_bins} equal-width bins")
    print(f"==============================")

    for b, sub in transitions_tmp.groupby("T_bin", observed=True):

        cm = pd.crosstab(
            sub["state"],
            sub["next_state"]
        ).reindex(
            index=state_order,
            columns=state_order,
            fill_value=0
        )

        cm_prob = cm.div(cm.sum(axis=1), axis=0)

        tmats_T[b] = {
            "counts": cm,
            "prob": cm_prob
        }

        rd_vals = compute_RD_from_transition_matrix(cm, cm_prob)

        row = {
            "n_bins": n_bins,
            "T_bin": str(b),
            "T_min": sub["T_corr_end"].min(),
            "T_max": sub["T_corr_end"].max(),
            "T_mid": 0.5 * (
                sub["T_corr_end"].min() + sub["T_corr_end"].max()
            ),
            "T_median": sub["T_corr_end"].median(),
            "n_transitions": len(sub),
        }

        row.update(rd_vals)
        all_RD_rows.append(row)

        print(f"\n{b}")
        print(cm_prob)

    all_tmats[n_bins] = tmats_T

    print("\nTransitions per bin:")
    print(transitions_tmp.groupby("T_bin", observed=True).size())

    print("\nTransitions per bin and state:")
    print(
        transitions_tmp
        .groupby(["T_bin", "state"], observed=True)["next_state"]
        .count()
    )

RD_bins_Tcorr_end = pd.DataFrame(all_RD_rows)

RD_bins_Tcorr_end

# Persistance and drift

In [ ]:
p_up = (transitions["delta"] > 0).mean()
p_same = (transitions["delta"] == 0).mean()
p_down = (transitions["delta"] < 0).mean()
mean_delta = transitions["delta"].mean()

p_up, p_same, p_down, mean_delta


### Mateix però per stop

In [ ]:
drift_by_stop = transitions.groupby("stop_idx").agg(
    n=("delta", "size"),
    p_up=("delta", lambda x: (x > 0).mean()),
    p_same=("delta", lambda x: (x == 0).mean()),
    p_down=("delta", lambda x: (x < 0).mean()),
    mean_delta=("delta", "mean"),
)

drift_by_stop.to_csv("csvs_proces/drift_by_stop.csv", index=False)
drift_by_stop

plt.figure(figsize=(8, 5))
plt.plot(drift_by_stop["p_up"], marker="o", label="p_up")
plt.plot(drift_by_stop["p_same"], marker="o", label="p_same")
plt.plot( drift_by_stop["p_down"], marker="o", label="p_down")

plt.xlabel("Stop d'origen")
plt.ylabel("Probabilitat")
plt.title("Probabilitats de transició per stop")
plt.legend()
plt.tight_layout()
plt.show()

# plot de la tendència: sembla que es tendeix cap a reduir el discomfort

In [ ]:
plt.figure(figsize=(7, 4))
plt.axhline(0, color="black", linestyle="--")
plt.plot( drift_by_stop["mean_delta"], marker="o")
plt.xlabel("Stop d'origen")
plt.ylabel("Mean delta")
plt.title("Drift mitjà per stop")
plt.tight_layout()
plt.show()

# Variables col·lectives (consens i "entropia")

In [ ]:
consent = occ.copy()
consent = consent[consent["stop_idx"] <= 5].copy()
pivot = (
    consent.pivot(index="stop_idx", columns="state", values="p")
       .reindex(columns=state_order)
       .fillna(0)
)

H = -(pivot * np.log(pivot.replace(0, np.nan))).sum(axis=1).fillna(0) / np.log(3)
C = 1 - H
D = pivot["uncomfortable"] + 2 * pivot["very uncomfortable"]
m = pivot["very uncomfortable"] - pivot["comfortable-neutral"]

collective = pd.DataFrame({
    "entropy_norm": H,
    "consensus": C,
    "weighted_discomfort": D,
    "order_param": m,
})

collective.to_csv("csvs_proces/collective_metrics_by_stop.csv", index=False)
collective

# Opinion metrix plots 

In [ ]:

data = df.copy()
data = data[data["stop_idx"] <= 5].copy()

occ = (
    data.groupby("stop_idx")["state"]
    .value_counts(normalize=True)
    .rename("p")
    .reset_index()
)

occ_pivot = (
    occ.pivot(index="stop_idx", columns="state", values="p")
    .reindex(columns=state_order)
    .fillna(0)
)

entropy_norm = -(occ_pivot * np.log(occ_pivot.replace(0, np.nan))).sum(axis=1).fillna(0) / np.log(len(state_order))
p_mode = occ_pivot.max(axis=1)

opinion_metrics = pd.DataFrame({
    "stop_idx": occ_pivot.index,
    "entropy_norm": entropy_norm.values,
    "p_mode": p_mode.values,
})

fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(opinion_metrics["stop_idx"], opinion_metrics["entropy_norm"], marker="o", label="Entropia normalitzada",color  = "blue")
ax1.set_xlabel("Stop")
ax1.set_ylabel("Entropia normalitzada")


ax2 = ax1.twinx()
ax2.plot(opinion_metrics["stop_idx"], opinion_metrics["p_mode"], marker="s", label="Proporció de la moda",color = "orange")
ax2.set_ylabel("Proporció de la moda")

ax1.set_title("Alineament / dispersió de les opinions per stop")
fig.tight_layout()
plt.legend()
plt.show()

## aquest plot tambpoc te massa sentit realment, era per veure si amb el cansanci o amb el temps la gent s'alinea

# Aquí no veiem res, fem per T

In [ ]:
df = df.copy()

df["temp_bin_10"] = pd.qcut(
    df["T_env"],
    q=10,
    duplicates="drop"
)
state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

occ_temp = (
    df.groupby("temp_bin_10")["state"]
    .value_counts(normalize=True)
    .rename("p")
    .reset_index()
)

occ_temp_pivot = (
    occ_temp.pivot(index="temp_bin_10", columns="state", values="p")
    .reindex(columns=state_order)
    .fillna(0)
)

entropy_temp = -(occ_temp_pivot * np.log(occ_temp_pivot.replace(0, np.nan))).sum(axis=1).fillna(0) / np.log(3)
consensus_temp = 1 - entropy_temp
p_mode_temp = occ_temp_pivot.max(axis=1)

entropy_by_temp = pd.DataFrame({
    "temp_bin_10": occ_temp_pivot.index.astype(str),
    "entropy_norm": entropy_temp.values,
    "consensus": consensus_temp.values,
    "p_mode": p_mode_temp.values,
    "n": df.groupby("temp_bin_10").size().values
})

entropy_by_temp.to_csv("csvs_proces/entropy_by_temp_10bins.csv", index=False)
entropy_by_temp

x = np.arange(len(entropy_by_temp))

fig, ax1 = plt.subplots(figsize=(10,5))


ax1.plot(x, entropy_by_temp["entropy_norm"], marker="o", label="Entropia normalitzada")
ax1.set_ylabel("Entropia normalitzada")
ax1.set_xlabel("Bin de temperatura")
ax1.set_xticks(x)
ax1.set_xticklabels(entropy_by_temp["temp_bin_10"], rotation=45, ha="right")

ax2 = ax1.twinx()
ax2.plot(x, entropy_by_temp["consensus"], marker="s", label="Consens",color = "orange")
ax2.set_ylabel("Consens")
plt.legend()

plt.title("Entropia / consens en funció de T")
fig.tight_layout()
plt.show()

In [ ]:
df = df.copy()

df["temp_bin_10"] = pd.qcut(
    df["T_env"],
    q=10,
    duplicates="drop"
)

state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

occ_temp = (
    df.groupby("temp_bin_10")["state"]
    .value_counts(normalize=True)
    .rename("p")
    .reset_index()
)

occ_temp_pivot = (
    occ_temp.pivot(index="temp_bin_10", columns="state", values="p")
    .reindex(columns=state_order)
    .fillna(0)
)

entropy_temp = -(occ_temp_pivot * np.log(occ_temp_pivot.replace(0, np.nan))).sum(axis=1).fillna(0) / np.log(3)
consensus_temp = 1 - entropy_temp
p_mode_temp = occ_temp_pivot.max(axis=1)

# Midpoints reals per l'eix X lineal
midpoints = occ_temp_pivot.index.map(lambda x: (x.left + x.right) / 2)

# DataFrame amb alineació per índex (segura)
counts = df.groupby("temp_bin_10").size().rename("n")

entropy_by_temp = pd.DataFrame({
    "entropy_norm": entropy_temp,
    "consensus": consensus_temp,
    "p_mode": p_mode_temp,
    "midpoint": midpoints,
}).join(counts)

entropy_by_temp.index = entropy_by_temp.index.astype(str)
entropy_by_temp = entropy_by_temp.reset_index(names="temp_bin_10")

entropy_by_temp.to_csv("csvs_proces/entropy_by_temp_10bins.csv", index=False)
entropy_by_temp

In [ ]:
x = entropy_by_temp["midpoint"]

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(x, entropy_by_temp["entropy_norm"], marker="o", label="Entropia normalitzada")
ax1.set_ylabel("Entropia normalitzada")
ax1.set_xlabel("Temperatura (°C)")
ax1.set_xticks(x)
ax1.set_xticklabels([f"{v:.1f}" for v in x], rotation=45, ha="right")

ax2 = ax1.twinx()
ax2.plot(x, entropy_by_temp["p_mode"], marker="s", label="p_mode (vot dominant)", color="orange")
ax2.set_ylabel("Fracció vot dominant")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

# n de cada bin com a anotació
for _, row in entropy_by_temp.iterrows():
    ax1.annotate(
        f"n={row['n']}",
        (row["midpoint"], row["entropy_norm"]),
        textcoords="offset points",
        xytext=(0, 6),
        fontsize=7,
        ha="center"
    )

plt.title("Entropia / consens en funció de T")
fig.tight_layout()
plt.show()

In [ ]:
rows_cumul = []

bins_sorted = entropy_by_temp.sort_values("midpoint")["temp_bin_10"].tolist()
midpoints_sorted = entropy_by_temp.sort_values("midpoint")["midpoint"].tolist()

for k, (bin_label, mid) in enumerate(zip(bins_sorted, midpoints_sorted)):
    # Selecciona tots els individus amb T >= T_k  (canvi clau)
    bins_from_here = bins_sorted[k:]
    subset = df[df["temp_bin_10"].astype(str).isin(bins_from_here)]

    p = subset["state"].value_counts(normalize=True).reindex(state_order).fillna(0)
    H = -(p * np.log(p.replace(0, np.nan))).sum() / np.log(3)
    p_mode = p.max()

    rows_cumul.append({
        "midpoint": mid,
        "temp_bin_10": bin_label,
        "entropy_norm_cumul": H,
        "consensus_cumul": 1 - H,
        "p_mode_cumul": p_mode,
        "n_cumul": len(subset)
    })

cumul_df = pd.DataFrame(rows_cumul)

In [ ]:
x = cumul_df["midpoint"]

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(x, cumul_df["entropy_norm_cumul"], marker="o", label="Entropia normalitzada", color="steelblue")
ax1.set_ylabel("Entropia normalitzada")
ax1.set_xlabel("Temperatura (°C)  —  població amb T ≥ T_k")
ax1.set_xticks(x)
ax1.set_xticklabels([f"{v:.1f}" for v in x], rotation=45, ha="right")

ax2 = ax1.twinx()
ax2.plot(x, cumul_df["p_mode_cumul"], marker="s", label="p_mode (vot dominant)", color="orange")
ax2.set_ylabel("Fracció vot dominant")

for _, row in cumul_df.iterrows():
    ax1.annotate(
        f"n={row['n_cumul']}",
        (row["midpoint"], row["entropy_norm_cumul"]),
        textcoords="offset points",
        xytext=(0, 6),
        fontsize=7,
        ha="center"
    )

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Entropia acumulada (T ≥ T_k) en funció de T")
fig.tight_layout()
plt.show()

In [ ]:
rows_cumul = []

bins_sorted = entropy_by_temp.sort_values("midpoint")["temp_bin_10"].tolist()
midpoints_sorted = entropy_by_temp.sort_values("midpoint")["midpoint"].tolist()

for k, (bin_label, mid) in enumerate(zip(bins_sorted, midpoints_sorted)):
    bins_from_here = bins_sorted[k:]
    subset = df[df["temp_bin_10"].astype(str).isin(bins_from_here)]

    n = len(subset)
    p = subset["state"].value_counts(normalize=True).reindex(state_order).fillna(0)
    H = -(p * np.log(p.replace(0, np.nan))).sum() / np.log(3)
    p_mode = p.max()

    # Bootstrap per estimar l'error
    n_boot = 500
    H_boot = []
    p_mode_boot = []

    for _ in range(n_boot):
        sample = subset["state"].sample(n=n, replace=True)
        p_b = sample.value_counts(normalize=True).reindex(state_order).fillna(0)
        H_b = -(p_b * np.log(p_b.replace(0, np.nan))).sum() / np.log(3)
        H_boot.append(H_b)
        p_mode_boot.append(p_b.max())

    rows_cumul.append({
        "midpoint": mid,
        "temp_bin_10": bin_label,
        "entropy_norm_cumul": H,
        "consensus_cumul": 1 - H,
        "p_mode_cumul": p_mode,
        "n_cumul": n,
        "entropy_err": np.std(H_boot),
        "p_mode_err": np.std(p_mode_boot),
    })

cumul_df = pd.DataFrame(rows_cumul)

In [ ]:
x = cumul_df["midpoint"]

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(x, cumul_df["entropy_norm_cumul"], marker="o", label="Entropia normalitzada", color="steelblue")
ax1.fill_between(
    x,
    cumul_df["entropy_norm_cumul"] - cumul_df["entropy_err"],
    cumul_df["entropy_norm_cumul"] + cumul_df["entropy_err"],
    alpha=0.2, color="steelblue"
)
ax1.set_ylabel("Entropia normalitzada")
ax1.set_xlabel("Temperatura (°C)  —  població amb T ≥ T_k")
ax1.set_xticks(x)
ax1.set_xticklabels([f"{v:.1f}" for v in x], rotation=45, ha="right")

ax2 = ax1.twinx()
ax2.plot(x, cumul_df["p_mode_cumul"], marker="s", label="p_mode (vot dominant)", color="orange")
ax2.fill_between(
    x,
    cumul_df["p_mode_cumul"] - cumul_df["p_mode_err"],
    cumul_df["p_mode_cumul"] + cumul_df["p_mode_err"],
    alpha=0.2, color="orange"
)
ax2.set_ylabel("Fracció vot dominant")

for _, row in cumul_df.iterrows():
    ax1.annotate(
        f"n={row['n_cumul']}",
        (row["midpoint"], row["entropy_norm_cumul"] + row["entropy_err"]),
        textcoords="offset points",
        xytext=(0, 4),
        fontsize=7,
        ha="center"
    )

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title("Entropia acumulada (T ≥ T_k) en funció de T  —  errors per bootstrap (n=500)")
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Copy data ---
df = df.copy()

# --- Define state order ---
state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

# --- Bin temperature into 10 quantiles ---
df["temp_bin_10"] = pd.qcut(
    df["<T-T_fixed+<T>>"],
    q=10,
    duplicates="drop"
)

# --- Compute state probabilities within each temperature bin ---
occ_temp = (
    df.groupby("temp_bin_10")["state"]
    .value_counts(normalize=True)
    .rename("p")
    .reset_index()
)

occ_temp_pivot = (
    occ_temp.pivot(index="temp_bin_10", columns="state", values="p")
    .reindex(columns=state_order)
    .fillna(0)
)

# --- Entropy, consensus ---
entropy_temp = -(
    occ_temp_pivot * np.log(occ_temp_pivot.replace(0, np.nan))
).sum(axis=1).fillna(0) / np.log(3)

consensus_temp = 1 - entropy_temp

# --- Order parameters ---
p_discomfort = (
    occ_temp_pivot["uncomfortable"]
    + occ_temp_pivot["very uncomfortable"]
)

severity_param = (
    occ_temp_pivot["uncomfortable"]
    + 2 * occ_temp_pivot["very uncomfortable"]
) / 2

# --- Build summary dataframe ---
summary_temp = pd.DataFrame({
    "temp_bin_10": occ_temp_pivot.index.astype(str),
    "entropy_norm": entropy_temp.values,
    "consensus": consensus_temp.values,
    "p_comfortable_neutral": occ_temp_pivot["comfortable-neutral"].values,
    "p_uncomfortable": occ_temp_pivot["uncomfortable"].values,
    "p_very_uncomfortable": occ_temp_pivot["very uncomfortable"].values,
    "p_discomfort": p_discomfort.values,
    "severity_param": severity_param.values,
    "n": df.groupby("temp_bin_10").size().values
})

# --- Find crossing: consensus ~= p_discomfort ---
summary_temp["cross_diff"] = summary_temp["consensus"] - summary_temp["p_discomfort"]
idx_cross = summary_temp["cross_diff"].abs().idxmin()

print("Bin closest to consensus = p_discomfort:")
print(summary_temp.loc[idx_cross, [
    "temp_bin_10", "consensus", "p_discomfort",
    "entropy_norm", "severity_param", "n"
]])

# --- Save table ---
summary_temp.to_csv("csvs_proces/consensus_vs_discomfort_temp_10bins.csv", index=False)

# --- Plot consensus vs discomfort probability ---
x = np.arange(len(summary_temp))

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    x,
    summary_temp["consensus"],
    marker="o",
    label="Consensus"
)
ax.plot(
    x,
    summary_temp["p_discomfort"],
    marker="s",
    label="P(discomfort)"
)

ax.axvline(idx_cross, linestyle=":", color="gray", linewidth=1,
           label="Closest crossing")
ax.set_xlabel("Temperature bin")
ax.set_ylabel("Value")
ax.set_xticks(x)
ax.set_xticklabels(summary_temp["temp_bin_10"], rotation=45, ha="right")
ax.legend(loc="best")

plt.title("Consensus vs discomfort probability by temperature bin")
fig.tight_layout()
plt.show()

# --- Optional: also plot severity instead of total discomfort ---
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    x,
    summary_temp["consensus"],
    marker="o",
    label="Consensus"
)
ax.plot(
    x,
    summary_temp["severity_param"],
    marker="^",
    label="Severity parameter"
)

ax.set_xlabel("Temperature bin")
ax.set_ylabel("Value")
ax.set_xticks(x)
ax.set_xticklabels(summary_temp["temp_bin_10"], rotation=45, ha="right")
ax.legend(loc="best")

plt.title("Consensus vs severity parameter by temperature bin")
fig.tight_layout()
plt.show()

summary_temp

# Consens per parada física? (consens social?)

In [ ]:
group_occ = (
    df.groupby(["walk_id", "stop_idx"])["state"]
    .value_counts(normalize=True)
    .rename("p")
    .reset_index()
)

group_occ_pivot = (
    group_occ.pivot(index=["walk_id", "stop_idx"], columns="state", values="p")
    .reindex(columns=state_order)
    .fillna(0)
)

group_entropy = -(group_occ_pivot * np.log(group_occ_pivot.replace(0, np.nan))).sum(axis=1).fillna(0) / np.log(3)
group_consensus = 1 - group_entropy
group_pmode = group_occ_pivot.max(axis=1)

group_metrics = group_occ_pivot.copy()
group_metrics["entropy_norm"] = group_entropy
group_metrics["consensus"] = group_consensus
group_metrics["p_mode"] = group_pmode
group_metrics = group_metrics.reset_index()

# mida del grup
group_n = df.groupby(["walk_id", "stop_idx"]).size().rename("n").reset_index()
group_metrics = group_metrics.merge(group_n, on=["walk_id", "stop_idx"], how="left")

group_metrics.to_csv("csvs_proces/group_consensus_walk_stop.csv", index=False)
group_metrics.head()

group_metrics = group_metrics[group_metrics["n"] >= 10].copy()
consensus_by_stop_within_walk = (
    group_metrics.groupby("stop_idx")
    .agg(
        mean_entropy=("entropy_norm", "mean"),
        mean_consensus=("consensus", "mean"),
        mean_pmode=("p_mode", "mean"),
        n_groups=("walk_id", "size"),
    )
    .reset_index()
)

consensus_by_stop_within_walk.to_csv("csvs_proces/consensus_by_stop_within_walk.csv", index=False)
consensus_by_stop_within_walk

### sembla que no ens diu res tampoc

# Mirem-ho les coses per règim d'HDX

In [ ]:
transitions["temp_bin"] = pd.qcut(
    transitions["<HDX-HDX_fixed+<HDX>>"],
    q=3,
    labels=["lower hdx", "neutral hdx", "higher hdx"]
)


## L'efecte es veu considerablement més amb HDX que no amb Temperatura

### Primer mirem les matrius de transició per règim

In [ ]:
tmats = {}
for b, sub in transitions.groupby("temp_bin"):
    cm = pd.crosstab(sub["state"], sub["next_state"]).reindex(
        index=state_order, columns=state_order, fill_value=0
    )
    tmats[b] = cm.div(cm.sum(axis=1), axis=0)


tmats

### I ara mirem el drift 

In [ ]:
drift_by_temp = transitions.groupby("temp_bin").agg(
    n=("delta", "size"),
    p_up=("delta", lambda x: (x > 0).mean()),
    p_same=("delta", lambda x: (x == 0).mean()),
    p_down=("delta", lambda x: (x < 0).mean()),
    mean_delta=("delta", "mean"),
)

drift_by_temp.to_csv("csvs_proces/drift_by_temp_bin.csv", index=False)
drift_by_temp

# Mesures de persistència

In [ ]:

tmp = df.sort_values(["ID", "stop_idx", "row_id"]).copy()
tmp["state_change"] = (tmp["state"] != tmp.groupby("ID")["state"].shift(1)).astype(int)
tmp["run_id"] = tmp.groupby("ID")["state_change"].cumsum()

runs = (
    tmp.groupby(["ID", "run_id"])
    .agg(
        state=("state", "first"),
        run_length=("row_id", "size"),
        first_stop=("stop_idx", "min"),
        last_stop=("stop_idx", "max"),
    )
    .reset_index()
)

runs.to_csv("csvs_proces/state_runs.csv", index=False)
runs.head()

run_summary = (
    runs.groupby("state")
    .agg(
        n_runs=("run_length", "size"),
        mean_run_length=("run_length", "mean"),
        median_run_length=("run_length", "median"),
        max_run_length=("run_length", "max"),
    )
    .reset_index()
)

run_summary.to_csv("csvs_proces/run_summary_by_state.csv", index=False)
run_summary

fig, ax = plt.subplots(figsize=(8,5))

for s in state_order:
    sub = runs[runs["state"] == s]
    counts = sub["run_length"].value_counts().sort_index()
    ax.plot(counts.index, counts.values, marker="o", label=s)

#ax.set_xscale("log")
#ax.set_yscale("log")
ax.set_xlabel("Longitud del run")
ax.set_ylabel("Nombre de runs")
ax.set_title("Distribució dels temps de residència per estat")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))

for s in state_order:
    sub = runs[runs["state"] == s]
    counts = sub["run_length"].value_counts().sort_index()
    ax.plot(counts.index, counts.values, marker="o", label=s)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Longitud del run")
ax.set_ylabel("Nombre de runs")
ax.set_title("Distribució dels temps de residència per estat")
ax.legend()
plt.tight_layout()
plt.show()

# Plot amb power law, exp i geom

In [ ]:
import numpy as np


def get_run_counts(runs, state_col="state", length_col="run_length"):
    out = {}
    for s, sub in runs.groupby(state_col):
        counts = sub[length_col].value_counts().sort_index()
        x = counts.index.to_numpy(dtype=float)
        y = counts.values.astype(float)
        out[s] = (x, y)
    return out

# -----------------------------------
# Simple visual fits
# -----------------------------------
def fit_power_law_visual(x, y):
    # y ~ A * x^{-alpha}
    lx = np.log(x)
    ly = np.log(y)
    slope, intercept = np.polyfit(lx, ly, 1)
    alpha = -slope
    A = np.exp(intercept)
    yhat = A * x**(-alpha)
    return {"alpha": alpha, "A": A, "yhat": yhat}

def fit_exponential_visual(x, y):
    # y ~ A * exp(-lambda x)
    ly = np.log(y)
    slope, intercept = np.polyfit(x, ly, 1)
    lam = -slope
    A = np.exp(intercept)
    yhat = A * np.exp(-lam * x)
    return {"lambda": lam, "A": A, "yhat": yhat}

def fit_geometric_from_pstay(x, y, p_stay=None):
    # y ~ C * (p_stay)^(x-1)
    # If p_stay is known from one-step persistence, use it directly.
    # Scale C to match the first empirical point.
    if p_stay is None:
        # visual fit from log(y) vs (x-1)
        slope, intercept = np.polyfit((x - 1), np.log(y), 1)
        p_stay = np.exp(slope)
    C = y[0]  # match first point
    yhat = C * (p_stay ** (x - 1))
    return {"p_stay": p_stay, "C": C, "yhat": yhat}

# -----------------------------------
# Main plotting function
# -----------------------------------
def plot_run_length_with_candidate_tails(
    runs,
    persistence_df=None,   # optional DataFrame with columns: state, p_stay
    state_col="state",
    length_col="run_length",
    title="Residence-time distributions with candidate tails"
):
    counts_dict = get_run_counts(runs, state_col=state_col, length_col=length_col)

    fig, axes = plt.subplots(1, len(counts_dict), figsize=(4.8 * len(counts_dict), 4.5), constrained_layout=True)
    if len(counts_dict) == 1:
        axes = [axes]

    pstay_map = {}
    if persistence_df is not None:
        pstay_map = dict(zip(persistence_df["state"], persistence_df["p_stay"]))

    for ax, (state, (x, y)) in zip(axes, counts_dict.items()):
        ax.loglog(x, y, "o-", label="Empirical")

        # power law
        pl = fit_power_law_visual(x, y)
        ax.loglog(x, pl["yhat"], "--", label=f"Power law (α={pl['alpha']:.2f})")

        # exponential
        ex = fit_exponential_visual(x, y)
        ax.loglog(x, ex["yhat"], "--", label=f"Exp (λ={ex['lambda']:.2f})")

        # geometric
        p0 = pstay_map.get(state, None)
        geo = fit_geometric_from_pstay(x, y, p_stay=p0)
        ax.loglog(x, geo["yhat"], "--", label=f"Geom (p={geo['p_stay']:.2f})")

        ax.set_title(str(state))
        ax.set_xlabel("Run length")
        ax.set_ylabel("Number of runs")
        ax.legend(fontsize=8)

    fig.suptitle(title, y=1.02)
    plt.show()

In [ ]:
comfort_order = [
  "comfortable-neutral",
  "uncomfortable",
  "very uncomfortable"
]

cm7_counts = pd.crosstab(
    transitions["state"],
    transitions["next_state"]
).reindex(index=comfort_order, columns=comfort_order, fill_value=0)

cm7_prob = cm7_counts.div(cm7_counts.sum(axis=1), axis=0)

persistence7 = pd.DataFrame({
    "state": comfort_order,
    "p_stay": [cm7_prob.loc[s, s] for s in comfort_order]
})


plot_run_length_with_candidate_tails(
    runs=runs,   # runs built for comfort7
    persistence_df=persistence7,
    state_col="state",
    length_col="run_length",
    title="Comfort7 residence-time distributions and candidate tails"
)

## Afegim unes pinzellades del ML calculat abans en règims binaris

In [ ]:
scores = df[df["stop_idx"] <= 5].copy()
scores_by_stop = scores.groupby("stop_idx").agg(
    vu_score_mean=("vu_score", "mean"),
    uw_score_mean=("uw_score", "mean"),
    suw_score_mean=("suw_score", "mean"),
)
scores_by_stop.to_csv("csvs_proces/scores_by_stop.csv", index=False)
scores_by_stop

### score abans de la transició

In [ ]:
df["next_state"] = df.groupby("ID")["state"].shift(-1)

pretransition = df[df["next_state"].notna()].groupby("next_state").agg(
    vu_score_mean=("vu_score", "mean"),
    uw_score_mean=("uw_score", "mean"),
    n=("row_id", "size"),
)
pretransition.to_csv("csvs_proces/pretransition_scores_by_next_state.csv", index=False)
pretransition

### per temperatura relativa

In [ ]:
transitions["hdx_bin"] = pd.qcut(
    transitions["<HDX-HDX_fixed+<HDX>>"],
    q=3,
    labels=["lower_hdx", "neutral_hdx", "higher_hdx"]
)

pretransition_hdx = (
    transitions.groupby(["hdx_bin", "next_state"])
    .agg(
        vu_score_mean=("vu_score", "mean"),
        uw_score_mean=("uw_score", "mean"),
        vu_prev_now=("is_very_uncomfortable", "mean"),
        uw_prev_now=("is_uncomfortable_or_worse", "mean"),
        n=("row_id", "size"),
    )
    .reset_index()
)

pretransition_hdx.to_csv("csvs_proces/pretransition_scores_by_hdx_bin_and_next_state.csv", index=False)
pretransition_hdx

# si afegim la caminada

In [ ]:


transitions["transition_type"] = np.select(
    [
        transitions["delta"] > 0,
        transitions["delta"] == 0,
        transitions["delta"] < 0,
    ],
    [
        "up",
        "same",
        "down",
    ],
    default="other"
)

walking_by_transition = (
    transitions.groupby(["transition_type", "thermal_comfort_walking"])
    .size()
    .rename("n")
    .reset_index()
)
walking_by_transition

In [ ]:
walking_map = {
    "Very comfortable": 0,
    "Comfortable": 0,
    "Slightly comfortable": 0,
    "Neutral": 0,
    "Slightly uncomfortable": 1,
    "Uncomfortable": 1,
    "Very uncomfortable": 2,
}

transitions["walking_num"] = transitions["thermal_comfort_walking"].map(walking_map)

transitions.groupby("transition_type")["walking_num"].agg(["mean", "median", "count"])

In [ ]:
transitions.groupby(["state", "transition_type"])["walking_num"].agg(["mean", "median", "count"])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

transition_order = ["down", "same", "up"]

walking_state_plot = (
    transitions.groupby(["state", "transition_type"])["walking_num"]
    .agg(mean="mean", median="median", count="count")
    .reset_index()
)

walking_state_plot["transition_type"] = pd.Categorical(
    walking_state_plot["transition_type"],
    categories=transition_order,
    ordered=True
)

walking_state_plot = walking_state_plot.sort_values(["state", "transition_type"])

x_map = {"down": 0, "same": 1, "up": 2}

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)

for state, sub in walking_state_plot.groupby("state"):
    x = sub["transition_type"].map(x_map).to_numpy()
    y = sub["mean"].to_numpy()
    ax.plot(x, y, marker="o", label=state)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["down", "same", "up"])
ax.set_xlabel("Transition type")
ax.set_ylabel("Mean walking_num")
ax.set_title("Walking comfort by transition type, conditioned on current state")
ax.legend()
plt.show()

In [ ]:
transitions["leave_cn"] = (
    (transitions["state"] == "comfortable-neutral") &
    (transitions["next_state"] != "comfortable-neutral")
).astype(int)

#transitions[transitions["state"] == "comfortable-neutral"].groupby(
  #  pd.qcut(transitions.loc[transitions["state"] == "comfortable-neutral", "walking_num"], q=3)
#)["leave_cn"].mean()


# Més non-eq analysis

In [ ]:
import numpy as np
import pandas as pd

eps = 1e-12
df_analysis = df[df["stop_idx"] <= 5].copy()

# --------------------------------------------------
# 1) Transition counts and transition matrix
# --------------------------------------------------
cm_counts = pd.crosstab(
    transitions["state"],
    transitions["next_state"]
).reindex(index=state_order, columns=state_order, fill_value=0)

P = cm_counts.div(
    cm_counts.sum(axis=1).replace(0, np.nan),
    axis=0
).fillna(0)

# --------------------------------------------------
# 2) Empirical distributions
# --------------------------------------------------

# A) source-state distribution (states that have an outgoing transition)
pi_source = (
    transitions["state"]
    .value_counts(normalize=True)
    .reindex(state_order, fill_value=0)
)

# B) full observed occupancy distribution
pi_occ = (
    df_analysis["comfort3_option1"]
    .value_counts(normalize=True)
    .reindex(state_order, fill_value=0)
)

# --------------------------------------------------
# 3) Stationary distribution implied by P
# Solve pi P = pi, sum(pi)=1
# --------------------------------------------------
A = P.T.values - np.eye(len(state_order))
A[-1, :] = 1.0
b = np.zeros(len(state_order))
b[-1] = 1.0

pi_stat = np.linalg.solve(A, b)
pi_stat = pd.Series(pi_stat, index=state_order, name="pi_stationary")

# numerical cleanup
pi_stat = pi_stat.clip(lower=0)
pi_stat = pi_stat / pi_stat.sum()

# --------------------------------------------------
# 4) Potential-like landscapes
# --------------------------------------------------
landscape = pd.DataFrame({
    "state": state_order,
    "pi_source": pi_source.values,
    "pi_occ": pi_occ.values,
    "pi_stat": pi_stat.reindex(state_order).values,
})

landscape["U_source"] = -np.log(np.clip(landscape["pi_source"], eps, None))
landscape["U_occ"] = -np.log(np.clip(landscape["pi_occ"], eps, None))
landscape["U_stat"] = -np.log(np.clip(landscape["pi_stat"], eps, None))

# optional summaries
valley_state_stat = landscape.loc[landscape["U_stat"].idxmin(), "state"]
valley_depth_stat = landscape["U_stat"].max() - landscape["U_stat"].min()

print("Stationary valley state:", valley_state_stat)
print("Stationary valley depth:", round(float(valley_depth_stat), 4))

landscape.to_csv("csvs_proces/landscape_comparison.csv", index=False)
landscape

In [ ]:
pi_vec = pi_stat.reindex(state_order).to_numpy(dtype=float)
P_mat = P.reindex(index=state_order, columns=state_order).to_numpy(dtype=float)

# pair flows F_ij = pi_i P_ij
flow = pi_vec[:, None] * P_mat

# antisymmetric probability currents
J = flow - flow.T

# use only unique pairs
upper_idx = np.triu_indices_from(J, k=1)
abs_J_pairs = np.abs(J[upper_idx])

# relative departure from detailed balance
eps = 1e-12
flow_pair_sum = flow + flow.T
rel_J = np.abs(J) / np.clip(flow_pair_sum, eps, None)
rel_J_pairs = rel_J[upper_idx]

db_summary = pd.DataFrame({
    "metric": [
        "mean_abs_db_departure",
        "max_abs_db_departure",
        "sum_abs_db_departure",
        "mean_rel_db_departure",
        "max_rel_db_departure",
    ],
    "value": [
        float(abs_J_pairs.mean()),
        float(abs_J_pairs.max()),
        float(abs_J_pairs.sum()),
        float(rel_J_pairs.mean()),
        float(rel_J_pairs.max()),
    ],
})

net_currents = []
for i, si in enumerate(state_order):
    for j, sj in enumerate(state_order):
        if i < j:
            fij = float(flow[i, j])
            fji = float(flow[j, i])
            Jij = float(J[i, j])
            net_currents.append({
                "state_i": si,
                "state_j": sj,
                "flow_ij": fij,
                "flow_ji": fji,
                "J_ij": Jij,
                "abs_J": abs(Jij),
                "rel_J": abs(Jij) / max(fij + fji, eps),
                "direction": f"{si}->{sj}" if Jij > 0 else f"{sj}->{si}",
            })

net_currents_df = pd.DataFrame(net_currents).sort_values("abs_J", ascending=False)
db_summary
net_currents_df

In [ ]:
# 3) Balanç detallat i corrent net
pi_vec = pi_stat.reindex(state_order).to_numpy(dtype=float)
P_mat = P.reindex(index=state_order, columns=state_order).to_numpy(dtype=float)

flow = np.outer(pi_vec, np.ones_like(pi_vec)) * P_mat
flow_rev = flow.T
J = flow - flow_rev  # corrent antisimmètric

db_abs_err = np.abs(J)
db_summary = pd.DataFrame({
    "metric": ["mean_abs_db_error", "max_abs_db_error"],
    "value": [float(db_abs_err.mean()), float(db_abs_err.max())],
})

net_currents = []
for i, si in enumerate(state_order):
    for j, sj in enumerate(state_order):
        if i < j:
            net_currents.append({
                "state_i": si,
                "state_j": sj,
                "J_ij": float(J[i, j]),
                "direction": f"{si}->{sj}" if J[i, j] > 0 else f"{sj}->{si}",
                "abs_J": float(abs(J[i, j]))
            })

net_currents_df = pd.DataFrame(net_currents).sort_values("abs_J", ascending=False)

db_summary.to_csv("csvs_proces/detailed_balance_summary.csv", index=False)
net_currents_df.to_csv("csvs_proces/net_currents_pairs.csv", index=False)

print(db_summary)
net_currents_df.head(10)


In [ ]:
# 4) Estat estacionari (pi* P = pi*) i comparació amb pi empírica
# Autovector dominant de P^T associat a lambda=1
vals, vecs = np.linalg.eig(P_mat.T)
idx = np.argmin(np.abs(vals - 1))
pi_star_raw = np.real(vecs[:, idx])

# Forcem no-negativitat i normalització
pi_star = np.abs(pi_star_raw)
if pi_star.sum() == 0:
    pi_star = np.ones_like(pi_star) / len(pi_star)
else:
    pi_star = pi_star / pi_star.sum()

stationary_cmp = pd.DataFrame({
    "state": state_order,
    "pi_empirical": pi_vec,
    "pi_stationary": pi_star,
})
stationary_cmp["abs_diff"] = np.abs(stationary_cmp["pi_empirical"] - stationary_cmp["pi_stationary"])

l1_gap = float(stationary_cmp["abs_diff"].sum())
kl_emp_star = float(np.sum(pi_vec * np.log(np.clip(pi_vec, 1e-12, None) / np.clip(pi_star, 1e-12, None))))

stationary_metrics = pd.DataFrame({
    "metric": ["L1_gap", "KL(emp||stationary)"],
    "value": [l1_gap, kl_emp_star]
})

stationary_cmp.to_csv("csvs_proces/stationary_vs_empirical.csv", index=False)
stationary_metrics.to_csv("csvs_proces/stationary_fit_metrics.csv", index=False)

print(stationary_metrics)
stationary_cmp


In [ ]:
# 5) SSD (Steepest-descent estocàstic): conques d'atracció i estat trampa
# Regla SSD simple: cada estat "baixa" cap al successor amb probabilitat màxima (incloent autolligam)
P_df = pd.DataFrame(P_mat, index=state_order, columns=state_order)
ssd_next = P_df.idxmax(axis=1)

# Detectar atractors seguint el mapping determinista

def find_attractor(start, nxt_map, max_iter=100):
    seen = []
    s = start
    for _ in range(max_iter):
        if s in seen:
            cyc = seen[seen.index(s):]
            return cyc[0], cyc
        seen.append(s)
        s = nxt_map[s]
    return seen[-1], [seen[-1]]

rows = []
for s in state_order:
    attr, cycle = find_attractor(s, ssd_next)
    rows.append({
        "state": s,
        "ssd_next": ssd_next[s],
        "attractor": attr,
        "cycle": " -> ".join(cycle),
        "is_fixed_point": len(cycle) == 1
    })

ssd_df = pd.DataFrame(rows)
basin_sizes = ssd_df.groupby("attractor").size().rename("basin_size").reset_index()

# Heurística d'estat trampa per l'estat extrem (últim de state_order)
extreme_state = state_order[-1]
p_stay_extreme = float(P_df.loc[extreme_state, extreme_state])
median_p_stay = float(np.median(np.diag(P_mat)))

incoming_minus_outgoing = 0.0
ext_idx = state_order.index(extreme_state)
for i in range(len(state_order)):
    if i == ext_idx:
        continue
    incoming_minus_outgoing += (flow[i, ext_idx] - flow[ext_idx, i])

trap_assessment = pd.DataFrame({
    "extreme_state": [extreme_state],
    "p_stay_extreme": [p_stay_extreme],
    "median_p_stay": [median_p_stay],
    "net_inflow_extreme": [float(incoming_minus_outgoing)],
    "acts_as_trap": [bool((p_stay_extreme >= median_p_stay) and (incoming_minus_outgoing > 0))]
})

ssd_df.to_csv("csvs_proces/ssd_state_mapping.csv", index=False)
basin_sizes.to_csv("csvs_proces/ssd_basin_sizes.csv", index=False)
trap_assessment.to_csv("csvs_proces/ssd_trap_assessment.csv", index=False)

display(ssd_df)
display(basin_sizes)
display(trap_assessment)
